# CMP symbolic regression — complete notebook using Colab versions

**Use a standard CPU runtime, then choose Runtime → Run all.** Authorize Google Drive when asked. No GPU, manual dataset upload, or code editing is required. Start with this notebook from the first cell. It uses Colab’s own Python and preinstalled scientific libraries; it installs PySR and missing backend dependencies into that same Python installation. No virtual environment, alternative Python interpreter, or uv setup is created. A new study folder keeps earlier notebooks’ checkpoints separate.

This notebook downloads the pinned PHM 2016 CMP dataset, audits it, creates three distinct evaluation protocols, tunes baselines, runs PySR with repeated seeds and ablations, checks equation stability, freezes every model, evaluates the holdouts, and saves tables, plots, equations, and predictions.

**Long-running by design:** the full configuration can take days of CPU time. Colab may disconnect or terminate the runtime. It cannot automatically restart a terminated Colab session. Reconnect and click **Run all** with the same study name: completed tasks are reused, and symbolic searches resume from the last saved chunk. Keep sufficient free space in Google Drive (several GB; checkpoint size varies).

### What is included

- Untouched source archive, SHA-256 manifest, strict joins, duplicate/identifier checks, missingness, and activity-interval diagnostics.
- Official benchmark, wafer-disjoint evaluation, and strictly chronological evaluation with boundary-crossing wafers purged.
- Mean/group mean, Ridge, Elastic Net, Huber, polynomial and spline regression, Random Forest, Extra Trees, histogram gradient boosting, and a pressure–rotation proxy.
- Main PySR, bounded operators, square-root operators, global equation, no usage counters, whole-run features, physics-proxy residual correction, and flagged-fitting-label sensitivity.
- Identical search budgets across symbolic ablations; baseline comparisons using the exact variables selected by SR.
- Equal fitting roles; no test-based hyperparameter choice; all primary labels retained.
- Repeated search seeds, wafer-bootstrap equation rediscovery, residual-deletion rediscovery, equation-domain/sensitivity checks, and portable prediction verification.
- Separate calibration data, prediction intervals, paired wafer bootstrap, temporal block bootstrap, subgroup errors, and input-range shift measurements.
- Persistent checkpoints, version checks, data/code/config fingerprints, frozen-model hashes, failure logs, PNG/PDF figures and a downloadable results archive.

The notebook does not fabricate results or a paper. It does not include a literature review. All metrics appear only after their experiments finish. A previous analysis has already examined this public dataset: the locks below enforce isolation within this run, not a claim that the dataset has never been seen.


## 1. Persistent storage and full study configuration

Defaults run the entire study. Keep `STUDY_NAME` unchanged when resuming. If you deliberately change methods or budgets, use a new name; the notebook refuses to mix incompatible checkpoints. Repeated runs on the same historical holdouts are exploratory reuse, not new independent validation.


In [ ]:
import os, sys, json, time, hashlib, subprocess, pathlib, shutil, csv, html
from pathlib import Path
from IPython.display import display, HTML, Image, FileLink

STUDY_NAME = "CMP_SR_Colab_native_full"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/CMP_SR_Research')
    LOCAL = Path('/content/cmp_sr_native_cache') / STUDY_NAME
    IN_COLAB = True
except ImportError:
    STORAGE = Path.cwd() / 'CMP_SR_Research'
    LOCAL = Path.cwd() / 'cmp_sr_native_cache' / STUDY_NAME
    IN_COLAB = False

STUDY = STORAGE / STUDY_NAME
STUDY.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "source_commit": "00d443e3f57379e3ad3d12c15a0738282b8210e5",
    "seed": 20260927,
    "protocols": ["official", "wafer_disjoint", "chronological"],
    "experiments": ["main", "bounded_operators", "sqrt_operators", "global",
                    "no_usage", "whole_run", "physics_correction", "flagged_fit_sensitivity"],
    "flag_threshold": 1000.0,
    "minimum_group_samples": 5,
    "cv_folds": 5,
    "workers": 2,
    "baseline_trials": 24,
    "baseline_shortlist": 3,
    "sr_configurations": [[4, 15, 0.001], [4, 25, 0.0001],
                          [8, 15, 0.001], [8, 25, 0.0001]],
    "tuning_seeds": [11, 22, 33],
    "final_seeds": [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010],
    "selection_tolerance": 0.05,
    "sr_iterations": 120,
    "sr_chunk_iterations": 10,
    "sr_populations": 8,
    "sr_population_size": 60,
    "sr_cycles": 200,
    "stability_bootstraps": 10,
    "bootstrap_draws": 5000,
    "interval_alpha": 0.10
}
config_path = STUDY / 'config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Settings differ from this study. Restore them to resume, or choose a new STUDY_NAME.')
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('Persistent study folder:', STUDY)
print('PySR iterations per search:', CONFIG['sr_iterations'])
print('Every symbolic ablation uses', len(CONFIG['tuning_seeds']), 'tuning seeds/configuration and', len(CONFIG['final_seeds']), 'final seeds.')


Mounted at /content/drive
Persistent study folder: /content/drive/MyDrive/CMP_SR_Research/CMP_SR_Colab_native_full
PySR iterations per search: 120
Every symbolic ablation uses 3 tuning seeds/configuration and 10 final seeds.


## 2. Use Colab's Python and library versions

This cell uses `sys.executable`, the same Python interpreter as the notebook. It records Colab’s installed versions and constrains pip to preserve them. It adds **PySR 2.5.0** and missing dependencies without replacing preinstalled NumPy, pandas, SciPy, scikit-learn, SymPy, or other dependency versions.

PySR needs Julia. The notebook uses Colab’s Julia when available; otherwise PySR installs its required backend automatically. The local cache and Drive archive below store Julia packages, not a separate Python environment. Analysis phases launch fresh processes using **the same Colab Python and packages**, so newly installed modules are available without reloading already imported kernel modules.

**Resuming:** retain the same `STUDY_NAME` and Colab runtime version. The notebook records the versions of added PySR dependencies and reinstalls those same versions when they are missing after a reconnect. Existing Colab libraries are preserved. If their versions change, the notebook stops before loading incompatible checkpoints. A package-installation conflict is reported clearly before the expensive study starts.


In [ ]:
import importlib.metadata as metadata, platform, re, tarfile

# Use exactly the interpreter running this Colab notebook.
PYTHON = Path(sys.executable)
PYSR_VERSION = '2.5.0'
CORE_PACKAGES = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'sympy',
                 'matplotlib', 'requests', 'cloudpickle', 'joblib']

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as source:
        for block in iter(lambda: source.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def atomic_text(path, text):
    path = Path(path)
    temp = path.with_name(path.name + '.pending')
    temp.write_text(text, encoding='utf-8')
    os.replace(temp, path)

def installed_versions():
    return {re.sub(r'[-_.]+', '-', d.metadata['Name']).lower(): d.version
            for d in metadata.distributions() if d.metadata.get('Name')}

if not IN_COLAB:
    raise RuntimeError('Open this notebook in a hosted Google Colab Python runtime.')
before = installed_versions()
missing_core = [name for name in CORE_PACKAGES if name not in before]
if missing_core:
    raise RuntimeError('This Colab runtime is missing standard libraries: '+str(missing_core)+
                       '. Select a standard Python runtime, then Run all.')

native_runtime = {'python':platform.python_version(),
                  'packages':{name:before[name] for name in CORE_PACKAGES}}
native_record = STUDY/'native_colab_runtime.json'
if native_record.exists():
    saved = json.loads(native_record.read_text())
    if saved != native_runtime:
        raise RuntimeError('Colab Python or scientific-library versions changed since this study started. '
                           'Select the original Colab runtime version to resume these checkpoints, '
                           'or use a new STUDY_NAME for a new study. Colab libraries will not be replaced.')
else:
    if not (STUDY/'colab_packages_before.json').exists():
        atomic_text(STUDY/'colab_packages_before.json', json.dumps(before, indent=2, sort_keys=True))

# Constraints preserve ALL installed dependencies, not only NumPy and pandas.
# Only the requested symbolic-regression engine may be installed/replaced.
additions_record = STUDY/'pysr_additions.json'
saved_additions = json.loads(additions_record.read_text()) if additions_record.exists() else {}
for name, version in saved_additions.items():
    if name != 'pysr' and name in before and before[name] != version:
        raise RuntimeError(f'{name} differs from this study: current {before[name]}, saved {version}. '
                           'Use the original runtime to resume. Existing packages will not be replaced.')
constraints = LOCAL/'preserve_colab_versions.txt'
preserved = {**before, **saved_additions}
atomic_text(constraints, '\n'.join(f'{name}=={version}' for name,version in sorted(preserved.items())
                                  if name != 'pysr')+'\n')
targets = ['pysr=='+PYSR_VERSION] + [f'{name}=={version}' for name,version in saved_additions.items()
                                   if name != 'pysr' and name not in before]
command = [str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input',
           '--retries', '4', '--timeout', '90', '--upgrade-strategy', 'only-if-needed',
           '--constraint', str(constraints)] + targets
print('Using Colab Python:', PYTHON, '|', platform.python_version(), flush=True)
print('Keeping Colab libraries:', native_runtime['packages'], flush=True)
try:
    subprocess.run(command, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('PySR installation failed. Read the pip message above. Existing Colab package '
                       'versions were constrained against replacement; if they are incompatible with '
                       'PySR 2.5.0, select the latest Colab runtime and rerun. Network failures can '
                       'be retried with Run all.') from exc
after = installed_versions()
changed = {name:(version,after.get(name)) for name,version in before.items()
           if name != 'pysr' and after.get(name) != version}
if changed:
    raise RuntimeError('Unexpected change to a preinstalled package: '+repr(changed))
if after.get('pysr') != PYSR_VERSION:
    raise RuntimeError('The requested PySR version was not installed.')
if saved_additions and any(after.get(name) != version for name,version in saved_additions.items()):
    raise RuntimeError('An added dependency does not match the saved checkpoint environment.')
if not additions_record.exists():
    original_packages = json.loads((STUDY/'colab_packages_before.json').read_text())
    atomic_text(additions_record, json.dumps({name:version for name,version in after.items()
                if name not in original_packages or name in {'pysr','juliacall','juliapkg'}},
                indent=2, sort_keys=True))
if not native_record.exists(): atomic_text(native_record, json.dumps(native_runtime, indent=2))
atomic_text(STUDY/'colab_packages_after.json', json.dumps(after, indent=2, sort_keys=True))
# This is an audit record, never an instruction to reinstall or downgrade Colab.
atomic_text(STUDY/'requirements.lock.txt',
            subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True))

WORKER_ENV = os.environ.copy()
WORKER_ENV.update({
    'PYTHONUNBUFFERED':'1', 'PYTHONHASHSEED':str(CONFIG['seed']),
    'OMP_NUM_THREADS':str(CONFIG['workers']), 'OPENBLAS_NUM_THREADS':'1', 'MKL_NUM_THREADS':'1',
    'JULIA_NUM_THREADS':'1', 'JULIA_NUM_GC_THREADS':'1', 'JULIA_NUM_PRECOMPILE_TASKS':'1',
    'JULIA_CPU_TARGET':'generic', 'PYTHON_JULIACALL_THREADS':'1',
    'PYTHON_JULIACALL_HANDLE_SIGNALS':'yes', 'PYTHON_JULIACALL_AUTOLOAD_IPYTHON_EXTENSION':'no',
    'JULIA_DEPOT_PATH':str(LOCAL/'julia_depot'), 'PYTHON_JULIAPKG_PROJECT':str(LOCAL/'julia_project'),
    'MPLCONFIGDIR':str(LOCAL/'matplotlib')
})
# Julia is required by PySR. Prefer the Julia already supplied by Colab.
julia_executable = shutil.which('julia')
if julia_executable:
    WORKER_ENV['PYTHON_JULIAPKG_EXE'] = julia_executable
    subprocess.run([julia_executable, '--startup-file=no', '--version'], check=True)
else:
    WORKER_ENV.pop('PYTHON_JULIAPKG_EXE', None)
    print('Colab has no Julia executable on PATH; PySR will install its required Julia backend automatically.')

# These directories cache the Julia backend only. They contain no Python environment.
JULIA_CACHE = STUDY/'julia_backend_cache.tar.gz'
JULIA_CACHE_META = JULIA_CACHE.with_suffix('.json')
if not (LOCAL/'julia_depot').exists() and JULIA_CACHE.exists() and JULIA_CACHE_META.exists():
    if sha256_file(JULIA_CACHE) != json.loads(JULIA_CACHE_META.read_text())['sha256']:
        raise RuntimeError('Julia cache checksum mismatch; preserve the study and investigate the archive.')
    print('Restoring the saved Julia backend cache...', flush=True)
    with tarfile.open(JULIA_CACHE, 'r:gz') as bundle:
        for member in bundle.getmembers():
            if not (LOCAL/member.name).resolve().is_relative_to(LOCAL.resolve()):
                raise RuntimeError('Unexpected cache path')
        if hasattr(tarfile, 'data_filter'): bundle.extractall(LOCAL, filter='data')
        else: bundle.extractall(LOCAL)

print('Ready: all analysis runs with Colab Python and its installed scientific libraries.')


Using Colab Python: /usr/bin/python3 | 3.13.15
Keeping Colab libraries: {'numpy': '1.26.4', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'sympy': '1.14.0', 'matplotlib': '3.10.0', 'requests': '2.32.4', 'cloudpickle': '3.1.2', 'joblib': '1.6.0'}
Ready: all analysis runs with Colab Python and its installed scientific libraries.


## 3. Complete analysis implementation

The following six cells contain all implementation code. They assemble one worker script in the study folder. You do not need external scripts or the earlier ZIP. Code is visible here for review; execution occurs in the subsequent numbered cells.

**Checkpoint policy:** every completed baseline configuration, selected model, and symbolic-search chunk is persisted. Checkpoint contents are hashed. An atomic pointer is committed only after a complete blob is written; the previous blob is retained for recovery. Never load checkpoint folders obtained from an untrusted source: model serialization is executable Python data.


In [ ]:
SOURCE_PARTS = {}


### Core utilities, configurations, atomic checkpoints, hashes and metrics.

In [ ]:
SOURCE_PARTS['01_core.py'] = '''"""CMP research pipeline. Embedded in the accompanying Colab notebook."""
import os, sys, json, time, hashlib, traceback, argparse, platform, subprocess
import io, zipfile, shutil, warnings, importlib.metadata, itertools, re, math
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import least_squares
import sympy as sp
import requests, cloudpickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, ParameterGrid, ParameterSampler
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ROOT = None
CFG = None
TARGET = 'AVG_REMOVAL_RATE'
KEY = ['WAFER_ID', 'STAGE']
PRESS = ['MAIN_OUTER_AIR_BAG_PRESSURE', 'CENTER_AIR_BAG_PRESSURE',
         'RETAINER_RING_PRESSURE', 'EDGE_AIR_BAG_PRESSURE',
         'RIPPLE_AIR_BAG_PRESSURE', 'PRESSURIZED_CHAMBER_PRESSURE']
ROT = ['WAFER_ROTATION', 'STAGE_ROTATION', 'HEAD_ROTATION']
SLURRY = ['SLURRY_FLOW_LINE_A', 'SLURRY_FLOW_LINE_B', 'SLURRY_FLOW_LINE_C']
USAGE = ['USAGE_OF_BACKING_FILM', 'USAGE_OF_DRESSER', 'USAGE_OF_POLISHING_TABLE',
         'USAGE_OF_DRESSER_TABLE', 'USAGE_OF_MEMBRANE', 'USAGE_OF_PRESSURIZED_SHEET']
CHANNELS = PRESS + ROT + SLURRY + USAGE
SHORT = dict(zip(CHANNELS, ['P_MAIN','P_CENTER','P_RING','P_EDGE','P_RIPPLE','P_CHAMBER',
                          'W_ROT','S_ROT','H_ROT','SL_A','SL_B','SL_C',
                          'U_FILM','U_DRESSER','U_TABLE','U_DTABLE','U_MEMBRANE','U_SHEET']))
META = {'SAMPLE_ID','WAFER_ID','STAGE','PARTITION',TARGET,'T_START','T_END',
        'MACHINE_ID','LONG_GAP','ACTIVITY_FALLBACK','N_CHAMBERS','FIRST_CHAMBER','SAME_TIME_COLLAPSED'}
EXPERIMENTS = {
    'main': dict(variant='active', grammar='arithmetic', routed=True, usage=True, residual=False, filtered=False),
    'bounded_operators': dict(variant='active', grammar='bounded', routed=True, usage=True, residual=False, filtered=False),
    'sqrt_operators': dict(variant='active', grammar='sqrt', routed=True, usage=True, residual=False, filtered=False),
    'global': dict(variant='active', grammar='arithmetic', routed=False, usage=True, residual=False, filtered=False),
    'no_usage': dict(variant='active', grammar='arithmetic', routed=True, usage=False, residual=False, filtered=False),
    'whole_run': dict(variant='whole', grammar='arithmetic', routed=True, usage=True, residual=False, filtered=False),
    'physics_correction': dict(variant='active', grammar='arithmetic', routed=True, usage=True, residual=True, filtered=False),
    'flagged_fit_sensitivity': dict(variant='active', grammar='arithmetic', routed=True, usage=True, residual=False, filtered=True),
}

def canonical(x):
    return json.dumps(x, sort_keys=True, separators=(',', ':'), default=lambda a: a.item() if hasattr(a,'item') else str(a))

def digest(x):
    return hashlib.sha256(canonical(x).encode()).hexdigest()

def filehash(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def atomic_bytes(path, blob):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.pending')
    with open(tmp,'wb') as f:
        f.write(blob); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)

def save_json(path, data):
    atomic_bytes(path,(json.dumps(data,indent=2,default=lambda x:x.item() if hasattr(x,'item') else str(x))+'\\n').encode())

def read_json(path):
    return json.loads(Path(path).read_text())

def save_csv(path, df):
    atomic_bytes(path,df.to_csv(index=False,float_format='%.17g').encode())

def read_csv(path):
    return pd.read_csv(path,float_precision='round_trip',dtype={'SAMPLE_ID':str,'WAFER_ID':str,'STAGE':str})

def journal(event, **kw):
    rec={'utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'event':event,**kw}
    with open(ROOT/'events.jsonl','a',encoding='utf-8') as f: f.write(canonical(rec)+'\\n')
    print(rec['utc'],event,canonical(kw),flush=True)

def store_state(folder, value):
    """Commit an immutable blob, then an atomic pointer. Retain previous blob too."""
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    data=cloudpickle.dumps(value); sha=hashlib.sha256(data).hexdigest()
    pointer=folder/'state.json'
    previous=read_json(pointer).get('sha') if pointer.exists() else None
    path=folder/(sha+'.pkl')
    if not path.exists(): atomic_bytes(path,data)
    save_json(pointer,{'sha':sha,'previous':previous})
    for old in folder.glob('*.pkl'):
        if old.stem not in {sha,previous}: old.unlink()

def load_state(folder):
    folder=Path(folder); pointer=folder/'state.json'
    if not pointer.exists(): return None
    meta=read_json(pointer)
    for sha in dict.fromkeys([meta['sha'],meta.get('previous')]):
        if not sha: continue
        p=folder/(sha+'.pkl')
        if p.exists() and filehash(p)==sha:
            if sha!=meta['sha']: journal('recovered_previous_checkpoint',path=str(folder))
            return cloudpickle.loads(p.read_bytes())
    raise RuntimeError(f'Checkpoint integrity failure: {folder}. Do not silently reuse it.')

def cached(path, fn):
    old=load_state(path)
    if old is not None and not (isinstance(old,dict) and old.get('failures')): return old
    out=fn(); store_state(path,out); return out

def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    if len(y)!=len(p) or not np.all(np.isfinite(p)):
        return dict(n=len(y),RMSE=None,MSE=None,MAE=None,R2=None,invalid_predictions=int((~np.isfinite(p)).sum()))
    return dict(n=len(y),MSE=float(mean_squared_error(y,p)),RMSE=float(np.sqrt(mean_squared_error(y,p))),
                MAE=float(mean_absolute_error(y,p)),R2=float(r2_score(y,p)) if len(y)>1 else None,invalid_predictions=0)

def checked_merge(a,b,on,how='inner'):
    return a.merge(b,on=on,how=how,validate='one_to_one')

def init(root,config):
    global ROOT,CFG
    ROOT=Path(root); ROOT.mkdir(parents=True,exist_ok=True); CFG=read_json(config)
    for p in ['data','audit','splits','models','sr','figures','tables','exports','logs']: (ROOT/p).mkdir(exist_ok=True)
    fingerprint={'config':CFG,'code_sha256':filehash(__file__)}
    lock=ROOT/'configuration.json'
    if lock.exists() and read_json(lock)!=fingerprint:
        raise RuntimeError('Code/config changed for this study. Use a new STUDY_NAME; never mix checkpoints.')
    if not lock.exists(): save_json(lock,fingerprint)
    versions={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','sympy','pysr','juliacall','juliapkg','cloudpickle','matplotlib']}
    env={'packages':versions,'python':platform.python_version(),'platform':platform.platform()}
    old=ROOT/'environment.json'
    if old.exists() and read_json(old)['packages']!=versions: raise RuntimeError('Package versions differ from checkpoints.')
    if old.exists() and read_json(old)['python']!=env['python']: raise RuntimeError('Python version differs from checkpoints.')
    if not old.exists(): save_json(old,env)
    np.random.seed(CFG['seed'])

def forbid_training_after_seal():
    if (ROOT/'FROZEN.json').exists(): raise RuntimeError('Models are frozen. Training is locked for this study.')

def feature_cols(df,usage=True,proxy=False):
    cols=[c for c in df if c not in META and pd.api.types.is_numeric_dtype(df[c])]
    if not usage: cols=[c for c in cols if not c.startswith('U_')]
    if proxy: cols=[c for c in cols if c.startswith('PV_') or c=='STAGE_B']
    return cols
'''


### Download, raw-data audit, transparent activity features, and frozen split assignments.

In [ ]:
SOURCE_PARTS['02_data.py'] = '''def download_data():
    archive=ROOT/'data'/'source.zip'
    commit=CFG['source_commit']
    url=f'https://codeload.github.com/akangel0307/PHM-Data-Challenge/zip/{commit}'
    marker=ROOT/'data'/'download.json'
    if marker.exists():
        info=read_json(marker)
        if archive.exists() and filehash(archive)==info['sha256']: return
    forbid_training_after_seal()
    for attempt in range(5):
        try:
            journal('download_start',attempt=attempt+1,url=url)
            tmp=archive.with_suffix('.partial')
            with requests.get(url,stream=True,timeout=(30,120)) as r:
                r.raise_for_status()
                with open(tmp,'wb') as f:
                    for chunk in r.iter_content(1024*1024): f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                if z.testzip() is not None: raise IOError('ZIP CRC verification failed')
            os.replace(tmp,archive)
            save_json(marker,{'url':url,'commit':commit,'sha256':filehash(archive),'bytes':archive.stat().st_size})
            return
        except (requests.RequestException,IOError,zipfile.BadZipFile) as e:
            journal('download_retry',error=str(e))
            if attempt==4: raise
            time.sleep(min(30,2**attempt))

def extract_data():
    dest=ROOT/'data'/'raw'; marker=ROOT/'data'/'extracted.json'
    if marker.exists(): return Path(read_json(marker)['base'])
    download_data(); dest.mkdir(exist_ok=True)
    rows=[]
    with zipfile.ZipFile(ROOT/'data'/'source.zip') as z:
        for info in z.infolist():
            # Only original challenge data; ignore repository code and processed datasets.
            if '/data/2016 PHM Data Challenge/' not in info.filename or info.is_dir(): continue
            if '__MACOSX' in info.filename: continue
            relative=info.filename.split('/data/2016 PHM Data Challenge/',1)[1]
            p=(dest/relative).resolve()
            if not p.is_relative_to(dest.resolve()): raise RuntimeError('Unsafe archive path')
            p.parent.mkdir(parents=True,exist_ok=True)
            atomic_bytes(p,z.read(info))
            rows.append({'path':relative,'bytes':p.stat().st_size,'sha256':filehash(p)})
    if not rows: raise RuntimeError('Expected original dataset directory absent from pinned repository')
    save_csv(ROOT/'audit'/'raw_manifest.csv',pd.DataFrame(rows))
    save_json(marker,{'base':str(dest),'n_files':len(rows),'manifest_sha':filehash(ROOT/'audit'/'raw_manifest.csv')})
    return dest

def locations():
    base=extract_data()
    data=base/'2016 PHM DATA CHALLENGE CMP DATA SET'
    val=base/'2016 PHM DATA CHALLENGE CMP VALIDATION DATA SET'
    return {'train':(data/'CMP-data'/'training',data/'CMP-training-removalrate.csv'),
            'val':(val/'validation',val/'CMP-validation-removalrate.csv'),
            'test':(data/'CMP-data'/'test',data/'CMP-test-removalrate.csv')}

def load_labels(part,keys_only=False):
    if part=='test' and not keys_only and not (ROOT/'FROZEN.json').exists():
        raise RuntimeError('Official test labels are locked until all protocols and experiments are frozen')
    path=locations()[part][1]
    y=pd.read_csv(path,usecols=KEY if keys_only else KEY+[TARGET],dtype={'WAFER_ID':str,'STAGE':str})
    y['WAFER_ID']=y.WAFER_ID.str.strip(); y['STAGE']=y.STAGE.str.strip()
    if y.duplicated(KEY).any(): raise ValueError(f'Duplicate label keys: {part}')
    if not keys_only and not np.isfinite(y[TARGET]).all(): raise ValueError('Nonfinite ground-truth label')
    y['PARTITION']=part; y['SAMPLE_ID']=part+'|'+y.WAFER_ID+'|'+y.STAGE
    return y

def activity_mask(frame):
    """Run-local, label-free high-activity heuristic, NOT a validated polishing detector.
    Normalize available nonconstant pressure/absolute rotation channels within chamber.
    Test trajectories may use their own complete signals: the task is post-process metrology.
    """
    signals=[]
    for c in PRESS+ROT:
        v=frame[c].to_numpy(float)
        if c in ROT: v=np.abs(v)
        lo,hi=np.nanquantile(v,[.1,.9])
        if hi-lo>1e-10:
            signals.append(np.clip((v-lo)/(hi-lo),0,1))
    if not signals: return np.ones(len(frame),bool),True
    score=np.nanmean(signals,axis=0); use=score>=0.5
    if use.sum()<5 or use.mean()>.95: return np.ones(len(frame),bool),True
    return use,False

def trajectory_features(g):
    g=g.sort_values(['CHAMBER','TIMESTAMP'],kind='stable')
    groups=set(pd.to_numeric(g.CHAMBER).astype(int).tolist())
    if groups.issubset({1,2,3}): chamber_group=1
    elif groups.issubset({4,5,6}): chamber_group=0
    else: raise ValueError(f'Unknown/mixed chamber group {groups}; cannot safely route')
    if g.MACHINE_ID.nunique()!=1: raise ValueError('Wafer-stage key spans different machines')
    common={'T_START':float(g.TIMESTAMP.min()),'T_END':float(g.TIMESTAMP.max()),
            'DURATION':float(g.TIMESTAMP.max()-g.TIMESTAMP.min()),'N_ROWS':len(g),
            'N_CHAMBERS':len(groups),'CHAMBER_GROUP':chamber_group,
            'STAGE_B':int(g.STAGE.iloc[0]=='B'),'MACHINE_ID':str(g.MACHINE_ID.iloc[0])}
    parts={'active':[],'whole':[]}; fallbacks=[]; long_gap=False; collapsed=0
    for _,ch in g.groupby(['CHAMBER','MACHINE_DATA'],sort=True):
        ch=ch.sort_values('TIMESTAMP',kind='stable').reset_index(drop=True)
        if ch.TIMESTAMP.duplicated().any():
            # Timestamp precision produces simultaneous updates in the released data.
            # Average coincident numeric readings explicitly, rather than taking an arbitrary last row.
            collapsed+=int(ch.TIMESTAMP.duplicated().sum())
            ch=ch.groupby('TIMESTAMP',as_index=False)[CHANNELS+['DRESSING_WATER_STATUS']].mean()
        t=ch.TIMESTAMP.to_numpy(float); dt=np.diff(t,append=t[-1])
        positive=dt[dt>0]; typical=float(np.median(positive)) if len(positive) else 1.
        cap=5*typical; long_gap=long_gap or bool((dt>cap).any())
        # Do not integrate across long acquisition gaps; report their presence.
        dt=np.where(dt>cap,0,dt)
        active,fallback=activity_mask(ch); fallbacks.append(fallback)
        for variant in parts:
            use=active if variant=='active' else np.ones(len(ch),bool)
            a=ch.loc[use]; w=dt[use]
            if w.sum()<=0: w=np.ones(len(a))
            f={'ACTIVE_FRAC':float(active.mean()),'OBSERVED_TIME':float(dt[use].sum())}
            for c in PRESS+ROT+SLURRY:
                v=a[c].to_numpy(float); ok=np.isfinite(v)&np.isfinite(w)
                if not ok.any(): mean=std=np.nan
                else:
                    ww=w[ok] if w[ok].sum()>0 else np.ones(ok.sum())
                    mean=float(np.average(v[ok],weights=ww)); std=float(np.sqrt(np.average((v[ok]-mean)**2,weights=ww)))
                f[SHORT[c]+'_MEAN']=mean; f[SHORT[c]+'_STD']=std
            for c in SLURRY: f[SHORT[c]+'_INT']=float(np.nansum(a[c].to_numpy(float)*dt[use]))
            f['DRESS_WATER']=float(pd.to_numeric(a.DRESSING_WATER_STATUS).mean())
            # Mean pressure proxy uses all pressure channels, avoiding unused P_MAIN alone.
            pressure=a[PRESS].mean(axis=1).to_numpy(float)
            for c in ROT: f['PV_'+SHORT[c]]=float(np.average(pressure*np.abs(a[c].to_numpy(float)),weights=w))
            for c in USAGE:
                f[SHORT[c]+'_START']=float(ch[c].iloc[0]); f[SHORT[c]+'_DELTA']=float(ch[c].iloc[-1]-ch[c].iloc[0])
            parts[variant].append(f)
    # Each chamber contributes equally; starts are averaged over chamber-specific starts.
    return {v:{**common,**pd.DataFrame(rows).mean().to_dict(),'ACTIVITY_FALLBACK':float(np.mean(fallbacks)),
               'LONG_GAP':int(long_gap),'SAME_TIME_COLLAPSED':collapsed} for v,rows in parts.items()}

def build_features():
    if (ROOT/'audit'/'features_complete.json').exists(): return
    forbid_training_after_seal(); extract_data(); audit={}; output={'active':[],'whole':[]}
    for part,(folder,_) in locations().items():
        cache=ROOT/'data'/('part_'+part)
        def build_one():
            arrays=[]; file_log=[]
            for f in sorted(folder.glob('*.csv')):
                try: d=pd.read_csv(f,dtype={'WAFER_ID':str,'STAGE':str})
                except pd.errors.EmptyDataError: d=pd.DataFrame()
                file_log.append({'file':f.name,'rows':len(d)})
                if len(d): d['SOURCE_FILE']=f.name; arrays.append(d)
            if not arrays: raise RuntimeError(f'No sensor records for {part}')
            x=pd.concat(arrays,ignore_index=True)
            required=set(KEY+CHANNELS+['CHAMBER','MACHINE_ID','MACHINE_DATA','TIMESTAMP','DRESSING_WATER_STATUS'])
            if not required.issubset(x): raise ValueError(f'Missing columns {required-set(x)}')
            x['WAFER_ID']=x.WAFER_ID.str.strip(); x['STAGE']=x.STAGE.str.strip()
            if not set(x.STAGE).issubset({'A','B'}): raise ValueError('Unexpected processing stage')
            identity=[c for c in x if c!='SOURCE_FILE']
            duplicated=int(x.duplicated(identity).sum()); x=x.drop_duplicates(identity)
            labels=load_labels(part,keys_only=True)
            rows={v:[] for v in output}
            for key,g in x.groupby(KEY,sort=True):
                ff=trajectory_features(g)
                for v,f in ff.items(): rows[v].append({**dict(zip(KEY,key)),**f})
            dfs={v:pd.DataFrame(r) for v,r in rows.items()}
            meta={'sensor_files':len(file_log),'empty_files':[r['file'] for r in file_log if not r['rows']],
                  'sensor_rows':len(x),'exact_duplicate_rows_removed':duplicated,
                  'missing_sensor_values':int(x.isna().sum().sum()),'machine_ids':sorted(x.MACHINE_ID.astype(str).unique().tolist())}
            for v,df in dfs.items():
                joined=checked_merge(labels,df,KEY)
                if len(joined)!=len(labels) or len(df)!=len(labels): raise ValueError('Unmatched labels or sensor runs')
                dfs[v]=joined
            meta.update(n_samples=len(labels),chambers_per_run=dfs['whole'].N_CHAMBERS.value_counts().to_dict(),
                        simultaneous_readings_averaged=int(dfs['whole'].SAME_TIME_COLLAPSED.sum()),
                        long_gap_runs=int(dfs['whole'].LONG_GAP.sum()),
                        activity_fallback_by_group=dfs['active'].groupby('CHAMBER_GROUP').ACTIVITY_FALLBACK.mean().to_dict())
            # Representative sensor trajectories come ONLY from development data.
            if part=='train':
                pieces=[]
                for group in [0,1]:
                    row=dfs['whole'][dfs['whole'].CHAMBER_GROUP==group].iloc[0]
                    pieces.append(x[(x.WAFER_ID==row.WAFER_ID)&(x.STAGE==row.STAGE)].assign(CHAMBER_GROUP=group))
                save_csv(ROOT/'data'/'representative_trajectories.csv',pd.concat(pieces))
            return dfs,meta
        dfs,meta=cached(cache,build_one); audit[part]=meta
        for v in output: output[v].append(dfs[v])
        journal('features_partition_complete',partition=part,n=meta['n_samples'])
    for v,dfs in output.items(): save_csv(ROOT/'data'/f'features_{v}.csv',pd.concat(dfs,ignore_index=True))
    labels=pd.concat([load_labels('train'),load_labels('val')],ignore_index=True)
    save_csv(ROOT/'data'/'development_labels.csv',labels)
    allf=pd.concat(output['whole'],ignore_index=True)
    sets={p:set(d.WAFER_ID) for p,d in allf.groupby('PARTITION')}
    audit['overlap']={a+'_'+b:len(sets[a]&sets[b]) for a,b in itertools.combinations(['train','val','test'],2)}
    audit['labels_flagged_not_deleted']=labels.groupby('PARTITION')[TARGET].apply(lambda x:int((x>CFG['flag_threshold']).sum())).to_dict()
    audit['data_terms']='Source repository MIT license does not establish original dataset redistribution permission. Archive excludes raw data.'
    save_json(ROOT/'audit'/'data_audit.json',audit)
    definitions=[]
    for c in allf:
        desc='Identifier/metadata; excluded from regression' if c in META else 'See trajectory_features for exact formula'
        if c.endswith('_START'): desc='Arithmetic mean of usage at first timestamp of each visited chamber; not a wear measurement'
        elif c.endswith('_DELTA'): desc='Mean across chambers of last minus first usage counter; resets retained and flagged through raw audit'
        elif c.endswith('_MEAN'): desc='Equal-chamber average of within-chamber time-weighted mean'
        elif c.endswith('_STD'): desc='Equal-chamber average of within-chamber time-weighted standard deviation'
        elif c.startswith('PV_'): desc='Within-chamber weighted mean of mean pressure channels times absolute rotation; dimensionless proxy'
        elif c.endswith('_INT'): desc='Equal-chamber average of left-Riemann integration; long gaps contribute zero'
        definitions.append({'feature':c,'definition':desc,'available':'post-process; before metrology','units':'organizer-scaled or count'})
    save_csv(ROOT/'audit'/'feature_dictionary.csv',pd.DataFrame(definitions))
    save_json(ROOT/'audit'/'features_complete.json',{'feature_hashes':{v:filehash(ROOT/'data'/f'features_{v}.csv') for v in output}})

def partition_validation(v):
    split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=CFG['seed'])
    a,b=next(split.split(v,groups=v.WAFER_ID))
    return v.iloc[a].SAMPLE_ID.tolist(),v.iloc[b].SAMPLE_ID.tolist()

def make_splits():
    if (ROOT/'splits'/'complete.json').exists(): return
    forbid_training_after_seal(); f=read_csv(ROOT/'data'/'features_whole.csv'); records=[]; reports={}
    for protocol in CFG['protocols']:
        d=f.copy(); d['role']='unused'
        if protocol in ['official','wafer_disjoint']:
            tr=d[d.PARTITION=='train']; va=d[d.PARTITION=='val']; te=d[d.PARTITION=='test']
            if protocol=='wafer_disjoint':
                va=va[~va.WAFER_ID.isin(te.WAFER_ID)]
                tr=tr[~tr.WAFER_ID.isin(set(va.WAFER_ID)|set(te.WAFER_ID))]
            selection,calibration=partition_validation(va)
            d.loc[d.SAMPLE_ID.isin(tr.SAMPLE_ID),'role']='fit'
            d.loc[d.SAMPLE_ID.isin(selection),'role']='select'
            d.loc[d.SAMPLE_ID.isin(calibration),'role']='calibrate'
            d.loc[d.SAMPLE_ID.isin(te.SAMPLE_ID),'role']='test'
        elif protocol=='chronological':
            pool=d[d.PARTITION.isin(['train','val'])]
            times=pool.groupby('WAFER_ID').agg(start=('T_START','min'),end=('T_END','max')).sort_values('start')
            boundaries=[float(times.start.iloc[int(len(times)*q)]) for q in [.60,.75,.85]]
            bounds=[-np.inf]+boundaries+[np.inf]
            for role,lo,hi in zip(['fit','select','calibrate','test'],bounds[:-1],bounds[1:]):
                # Purge any wafer whose stages/runs cross a temporal boundary.
                ids=times.index[(times.start>=lo)&(times.end<hi)]
                d.loc[d.SAMPLE_ID.isin(pool[pool.WAFER_ID.isin(ids)].SAMPLE_ID),'role']=role
            reports[protocol]={'time_boundaries':boundaries,'purged_boundary_samples':int((d.loc[pool.index,'role']=='unused').sum())}
        else: raise ValueError(protocol)
        d['protocol']=protocol
        role_ids={r:set(g.WAFER_ID) for r,g in d[d.role!='unused'].groupby('role')}
        overlap={a+'_'+b:len(role_ids[a]&role_ids[b]) for a,b in itertools.combinations(role_ids,2)}
        if protocol!='official' and any(overlap.values()): raise AssertionError('Wafer overlap in disjoint protocol')
        if protocol=='chronological':
            for a,b in zip(['fit','select','calibrate'],['select','calibrate','test']):
                assert d.loc[d.role==a,'T_END'].max()<d.loc[d.role==b,'T_START'].min()
        for role in ['fit','select','calibrate','test']:
            counts=d[d.role==role].groupby('CHAMBER_GROUP').size()
            if len(counts)!=2 or counts.min()<CFG['minimum_group_samples']:
                raise RuntimeError(f'{protocol}/{role}: insufficient group samples {counts.to_dict()}')
        reports.setdefault(protocol,{}).update(role_counts=d.role.value_counts().to_dict(),wafer_overlap=overlap,
            group_counts=d.groupby(['role','CHAMBER_GROUP']).size().to_dict())
        records.append(d[['SAMPLE_ID','WAFER_ID','STAGE','PARTITION','T_START','T_END','CHAMBER_GROUP','protocol','role']])
    splits=pd.concat(records,ignore_index=True)
    save_csv(ROOT/'splits'/'assignments.csv',splits)
    # JSON-safe tuple-key conversion.
    for p in reports: reports[p]['group_counts']={str(k):v for k,v in reports[p]['group_counts'].items()}
    save_json(ROOT/'splits'/'report.json',reports)
    save_json(ROOT/'splits'/'complete.json',{'sha256':filehash(ROOT/'splits'/'assignments.csv')})

def frame(protocol,role,variant='active',labels=True):
    assignments=read_csv(ROOT/'splits'/'assignments.csv')
    ids=assignments[(assignments.protocol==protocol)&(assignments.role==role)].SAMPLE_ID
    x=read_csv(ROOT/'data'/f'features_{variant}.csv'); x=x[x.SAMPLE_ID.isin(ids)].reset_index(drop=True)
    if not labels: return x
    if role=='test' and not (ROOT/'FROZEN.json').exists(): raise RuntimeError('Evaluation labels locked')
    y=read_csv(ROOT/'data'/'development_labels.csv')[['SAMPLE_ID',TARGET]]
    if (x.PARTITION=='test').any(): y=pd.concat([y,load_labels('test')[['SAMPLE_ID',TARGET]]],ignore_index=True)
    out=checked_merge(x,y,['SAMPLE_ID'])
    assert len(out)==len(x)
    return out
'''


### Grouped/forward CV, tuned baselines, and fitting-only feature ranking.

In [ ]:
SOURCE_PARTS['03_baselines.py'] = '''BASE_NAMES=['Mean','GroupMean','Ridge','ElasticNet','Huber','Poly2Ridge','SplineRidge',
            'RandomForest','ExtraTrees','HistGradientBoosting','PrestonProxy']

def cv_indices(d,protocol):
    """CV never contains selection, calibration or evaluation rows."""
    if protocol!='chronological':
        folds=min(CFG['cv_folds'],d.WAFER_ID.nunique())
        return list(GroupKFold(n_splits=folds).split(d,groups=d.WAFER_ID))
    times=d.groupby('WAFER_ID').agg(start=('T_START','min'),end=('T_END','max')).sort_values('start')
    blocks=np.array_split(times.index.to_numpy(),CFG['cv_folds']+1); out=[]
    for j in range(1,len(blocks)):
        valid_ids=blocks[j]; lo=times.loc[valid_ids,'start'].min()
        train_ids=times.index[times.end<lo]
        tr=np.flatnonzero(d.WAFER_ID.isin(train_ids)); va=np.flatnonzero(d.WAFER_ID.isin(valid_ids))
        if len(tr)>=10 and len(va)>=3:
            assert d.iloc[tr].T_END.max()<d.iloc[va].T_START.min()
            out.append((tr,va))
    if len(out)<2: raise ValueError('Insufficient forward-validation blocks')
    return out

def input_cols(name,d):
    if name=='PrestonProxy': return feature_cols(d,proxy=True)
    if name in ['Poly2Ridge','SplineRidge']:
        names=['STAGE_B','CHAMBER_GROUP','U_DTABLE_START','U_DRESSER_START','U_TABLE_START',
               'P_MAIN_MEAN','P_CENTER_MEAN','W_ROT_MEAN','SL_A_MEAN']
        return [c for c in names if c in d]
    return feature_cols(d)

def estimator(name,params):
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    seed=CFG['seed']
    if name in ['Mean','GroupMean']: return make_pipeline(imp,DummyRegressor(strategy='mean'))
    if name in ['Ridge','PrestonProxy']: return make_pipeline(imp,StandardScaler(),Ridge(**params))
    if name=='ElasticNet': return TransformedTargetRegressor(regressor=make_pipeline(imp,StandardScaler(),ElasticNet(max_iter=100000,**params)),transformer=StandardScaler())
    if name=='Huber': return TransformedTargetRegressor(regressor=make_pipeline(imp,StandardScaler(),HuberRegressor(max_iter=5000,**params)),transformer=StandardScaler())
    if name=='Poly2Ridge': return make_pipeline(imp,StandardScaler(),PolynomialFeatures(2,include_bias=False),StandardScaler(),Ridge(**params))
    if name=='SplineRidge':
        return make_pipeline(imp,SplineTransformer(n_knots=params['n_knots'],degree=3,extrapolation='constant'),StandardScaler(),Ridge(alpha=params['alpha']))
    if name=='RandomForest': return make_pipeline(imp,RandomForestRegressor(n_estimators=300,n_jobs=CFG['workers'],random_state=seed,**params))
    if name=='ExtraTrees': return make_pipeline(imp,ExtraTreesRegressor(n_estimators=300,n_jobs=CFG['workers'],random_state=seed,**params))
    if name=='HistGradientBoosting': return make_pipeline(imp,HistGradientBoostingRegressor(early_stopping=False,random_state=seed,**params))
    raise ValueError(name)

def parameter_space(name):
    if name in ['Mean','GroupMean']: return [{}]
    if name in ['Ridge','PrestonProxy','Poly2Ridge']: space={'alpha':list(np.logspace(-5,6,12))}
    elif name=='ElasticNet': space={'alpha':list(np.logspace(-4,2,7)),'l1_ratio':[.1,.5,.9,1.]}
    elif name=='Huber': space={'epsilon':[1.1,1.35,1.75,2.0],'alpha':[1e-5,.001,.1,10.]}
    elif name=='SplineRidge': space={'n_knots':[3,5,8,12],'alpha':[.001,.1,10.,1000.]}
    elif name in ['RandomForest','ExtraTrees']: space={'max_depth':[None,5,10,20],'min_samples_leaf':[1,3,8,20],'max_features':[.4,.7,1.]}
    else: space={'max_iter':[300,800,1600],'learning_rate':[.02,.05,.1],'max_leaf_nodes':[7,15,31],
                 'min_samples_leaf':[5,15,30],'l2_regularization':[0.,1.,10.]}
    candidates=list(ParameterGrid(space))
    n=CFG['baseline_trials']
    return candidates if len(candidates)<=n else list(ParameterSampler(space,n_iter=n,random_state=CFG['seed']))

def train_baseline(name,fit,select,protocol,folder):
    cols=input_cols(name,fit)
    if name=='GroupMean':
        values=fit.groupby(['CHAMBER_GROUP','STAGE_B'])[TARGET].mean()
        return {'kind':'groupmean','lookup':{f'{int(a)}|{int(b)}':float(v) for (a,b),v in values.items()},
                'default':float(fit[TARGET].mean()),'features':['CHAMBER_GROUP','STAGE_B'],'params':{}}
    scores=[]
    folds=cv_indices(fit,protocol)
    for params in parameter_space(name):
        candidate_id=digest(params)[:12]
        def score_one():
            errors=[]; fail=[]
            for tr,va in folds:
                try:
                    model=estimator(name,params).fit(fit.iloc[tr][cols],fit.iloc[tr][TARGET])
                    p=model.predict(fit.iloc[va][cols]); errors.extend((p-fit.iloc[va][TARGET].to_numpy())**2)
                except Exception as e: fail.append(type(e).__name__+': '+str(e))
            return {'params':params,'cv_mse':float(np.mean(errors)) if errors and not fail else float('inf'),'failures':fail}
        scores.append(cached(folder/'cv'/candidate_id,score_one))
    finite=[s for s in scores if np.isfinite(s['cv_mse'])]
    if not finite: raise RuntimeError(f'Every {name} configuration failed: {scores}')
    # Same fitting rows for all methods; selection rows choose among CV-shortlisted settings.
    options=[]
    for row in sorted(finite,key=lambda r:r['cv_mse'])[:CFG['baseline_shortlist']]:
        m=estimator(name,row['params']).fit(fit[cols],fit[TARGET])
        score=metrics(select[TARGET],m.predict(select[cols]))['RMSE']
        options.append((score,m,row))
    score,model,row=min(options,key=lambda v:v[0])
    save_json(folder/'search.json',{'trials':scores,'selected':row,'selection_rmse':score,
              'note':'CV scores are tuning estimates, not unbiased final performance estimates.'})
    return {'kind':'sklearn','estimator':model,'features':cols,'params':row['params'],'selection_rmse':score}

def predict_base(bundle,d):
    if bundle['kind']=='groupmean':
        return np.array([bundle['lookup'].get(f'{int(a)}|{int(b)}',bundle['default']) for a,b in zip(d.CHAMBER_GROUP,d.STAGE_B)])
    if bundle['kind']=='routed':
        p=np.full(len(d),np.nan)
        for g,b in bundle['groups'].items():
            use=(d.CHAMBER_GROUP==int(g)).to_numpy(); p[use]=predict_base(b,d.loc[use])
        return p
    return bundle['estimator'].predict(d[bundle['features']])

def base_path(protocol,name,mode='routed',filtered=False):
    return ROOT/'models'/protocol/('flagged_fit_sensitivity' if filtered else 'primary')/(name+'_'+mode)

def fit_baselines():
    if (ROOT/'models'/'complete.json').exists(): return
    forbid_training_after_seal()
    for protocol in CFG['protocols']:
        fit=frame(protocol,'fit'); select=frame(protocol,'select')
        for filtered in [False,True]:
            used=fit[fit[TARGET]<=CFG['flag_threshold']].copy() if filtered else fit.copy()
            names=BASE_NAMES if not filtered else ['Ridge','ExtraTrees','HistGradientBoosting','PrestonProxy']
            for name in names:
                modes=['global'] if name=='Mean' else ['routed'] if name=='GroupMean' else ['global','routed'] if not filtered else ['routed']
                for mode in modes:
                    folder=base_path(protocol,name,mode,filtered)
                    if load_state(folder/'final') is not None: continue
                    journal('baseline_start',protocol=protocol,name=name,mode=mode,filtered=filtered)
                    groups={}
                    for g in ([0,1] if mode=='routed' and name!='GroupMean' else [None]):
                        tr=used if g is None else used[used.CHAMBER_GROUP==g]
                        va=select if g is None else select[select.CHAMBER_GROUP==g]
                        groups[str(g)]=cached(folder/('group_'+str(g))/'chosen',lambda n=name,t=tr,v=va,f=folder/('group_'+str(g)):train_baseline(n,t,v,protocol,f))
                    bundle=groups['None'] if 'None' in groups else {'kind':'routed','groups':groups}
                    bundle.update(name=name,mode=mode,protocol=protocol,filtered=filtered,
                                  fit_ids=sorted(used.SAMPLE_ID),selection_ids=sorted(select.SAMPLE_ID))
                    store_state(folder/'final',bundle)
                    journal('baseline_complete',protocol=protocol,name=name,mode=mode)
    save_json(ROOT/'models'/'complete.json',{'complete':True})

def rank_features(fit,protocol,cols,folder):
    """Ranking derives only from fitting rows; excluded labels never enter sensitivity ranking."""
    def run():
        imp=np.zeros(len(cols)); folds=cv_indices(fit,protocol)
        for tr,va in folds:
            model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),
                                ExtraTreesRegressor(n_estimators=200,min_samples_leaf=3,random_state=CFG['seed'],n_jobs=CFG['workers']))
            model.fit(fit.iloc[tr][cols],fit.iloc[tr][TARGET])
            pi=permutation_importance(model,fit.iloc[va][cols],fit.iloc[va][TARGET],
                                      n_repeats=5,random_state=CFG['seed'],scoring='neg_mean_squared_error',n_jobs=1)
            imp+=pi.importances_mean/len(folds)
        out=pd.DataFrame({'feature':cols,'importance':imp}).sort_values('importance',ascending=False)
        save_csv(folder/'ranking.csv',out)
        return out.feature.tolist()
    return cached(folder/'ranking_state',run)
'''


### PySR search, warm-start resume, equal-budget ablations, and engine preflight.

In [ ]:
SOURCE_PARTS['04_symbolic.py'] = '''def sqrtabs_sympy(x): return sp.sqrt(sp.Abs(x))

def grammar_options(name):
    if name=='arithmetic': return {'binary_operators':['+','-','*'],'unary_operators':[]}
    if name=='bounded': return {'binary_operators':['+','-','*'],'unary_operators':['tanh'],
                               'nested_constraints':{'tanh':{'tanh':0}}}
    if name=='sqrt': return {'binary_operators':['+','-','*'],'unary_operators':['sqrtabs(x) = sqrt(abs(x))'],
                            'extra_sympy_mappings':{'sqrtabs':sqrtabs_sympy}}
    if name=='rational': return {'binary_operators':['+','-','*','/'],'unary_operators':[], 'constraints':{'/':(-1,7)}}
    raise ValueError(name)

def sr_grid():
    # Every experiment uses exactly the same number of configurations and seeds.
    return [{'k':k,'maxsize':size,'parsimony':p} for k,size,p in CFG['sr_configurations']]

def preprocess(fit,cols,target):
    x=fit[cols].replace([np.inf,-np.inf],np.nan)
    median=x.median().fillna(0).to_numpy(float); xx=x.fillna(dict(zip(cols,median))).to_numpy(float)
    mu=xx.mean(axis=0); sd=xx.std(axis=0); sd[sd<1e-12]=1.
    sy=float(np.std(target)); sy=sy if sy>1e-12 else 1.
    return {'features':cols,'median':median.tolist(),'mu':mu.tolist(),'sd':sd.tolist(),
            'y_mu':float(np.mean(target)),'y_sd':sy,
            'raw_min':np.min(xx,axis=0).tolist(),'raw_max':np.max(xx,axis=0).tolist()}

def transform(d,prep):
    x=d[prep['features']].to_numpy(float).copy()
    x=np.where(np.isfinite(x),x,np.asarray(prep['median']))
    return (x-np.asarray(prep['mu']))/np.asarray(prep['sd'])

@lru_cache(maxsize=8192)
def expression_callable(expr,n):
    symbols=sp.symbols(' '.join(f'z{i}' for i in range(n)),seq=True)
    expression=sp.sympify(expr,locals={str(s):s for s in symbols})
    if expression.free_symbols-set(symbols): raise ValueError('Equation contains unknown variables')
    return sp.lambdify(symbols,expression,modules='numpy'),expression

def expr_predict(expr,z):
    fn,_=expression_callable(expr,z.shape[1])
    with np.errstate(all='ignore'):
        p=np.asarray(fn(*[z[:,i] for i in range(z.shape[1])]),float)
        return np.broadcast_to(p,(len(z),)).copy()

def equation_predict(eq,d):
    p=eq['prep']; z=transform(d,p)
    return p['y_mu']+p['y_sd']*expr_predict(eq['expression'],z)

def choose_equation(candidates):
    candidates=[c for c in candidates if np.isfinite(c['selection_rmse']) and np.isfinite(c['train_rmse'])]
    if not candidates: raise RuntimeError('No finite symbolic candidates; run marked failed, no model substitution')
    threshold=min(c['selection_rmse'] for c in candidates)*(1+CFG['selection_tolerance'])
    eligible=[c for c in candidates if c['selection_rmse']<=threshold]
    return min(eligible,key=lambda c:(c['complexity'],c['selection_rmse'],c['expression']))

def sr_search(fit,selection,cols,params,seed,grammar,folder,base=None):
    """Checkpoint after each warm-start chunk; selection never uses final holdout labels."""
    result=folder/'result.json'
    if result.exists(): return read_json(result)
    from pysr import PySRRegressor
    yf=fit[TARGET].to_numpy(float).copy(); ys=selection[TARGET].to_numpy(float).copy()
    if base is not None:
        yf-=predict_base(base,fit); ys-=predict_base(base,selection)
    prep=preprocess(fit,cols,yf); xf=transform(fit,prep); xs=transform(selection,prep)
    yz=(yf-prep['y_mu'])/prep['y_sd']; names=[f'z{i}' for i in range(len(cols))]
    signature=digest({'fit_ids':fit.SAMPLE_ID.tolist(),'select_ids':selection.SAMPLE_ID.tolist(),
                      'fit_y':yf.tolist(),'select_y':ys.tolist(),'prep':prep,'params':params,'grammar':grammar,'seed':seed})
    state=load_state(folder/'checkpoint')
    if state is not None:
        if state['signature']!=signature: raise RuntimeError('SR checkpoint belongs to different data/settings')
        model=state['model']; done=state['done']
        if grammar=='sqrt': model.set_params(extra_sympy_mappings={'sqrtabs':sqrtabs_sympy})
    else:
        model=PySRRegressor(niterations=CFG['sr_chunk_iterations'],populations=CFG['sr_populations'],
             population_size=CFG['sr_population_size'],ncycles_per_iteration=CFG['sr_cycles'],
             maxsize=int(params['maxsize']),maxdepth=int(params['maxsize']),parsimony=float(params['parsimony']),
             random_state=int(seed),deterministic=True,parallelism='serial',precision=64,
             warm_start=True,progress=False,verbosity=0,model_selection='accuracy',
             batching=False,annealing=True,crossover_probability=0.2,input_stream='devnull',
             output_directory=str(folder/'native'),run_id='search',**grammar_options(grammar))
        done=0
    t0=time.time()
    while done<CFG['sr_iterations']:
        step=min(CFG['sr_chunk_iterations'],CFG['sr_iterations']-done)
        if grammar=='sqrt': model.set_params(extra_sympy_mappings={'sqrtabs':sqrtabs_sympy})
        model.set_params(niterations=step)
        model.fit(xf,yz,variable_names=names)
        done+=step
        store_state(folder/'checkpoint',{'signature':signature,'model':model,'done':done})
        save_json(folder/'progress.json',{'iterations_done':done,'iterations_total':CFG['sr_iterations'],'seed':seed})
        journal('sr_checkpoint',task=str(folder.relative_to(ROOT)),iterations=done)
    candidates=[]
    for idx,r in model.equations_.iterrows():
        expression=str(r['sympy_format'])
        pf=prep['y_mu']+prep['y_sd']*expr_predict(expression,xf)
        ps=prep['y_mu']+prep['y_sd']*expr_predict(expression,xs)
        if not np.all(np.isfinite(pf)) or not np.all(np.isfinite(ps)): continue
        native=prep['y_mu']+prep['y_sd']*model.predict(xf,index=int(idx))
        if not np.allclose(pf,native,rtol=1e-8,atol=1e-7): raise AssertionError('Symbolic export differs from PySR predictions')
        expression_obj=expression_callable(expression,len(cols))[1]
        used=sorted(str(s) for s in expression_obj.free_symbols)
        candidates.append({'expression':expression,'complexity':int(r['complexity']),
                           'symbolic_ops':int(sp.count_ops(expression_obj)), 'variables':used,
                           'seed':int(seed),'params':params,'prep':prep,
                           'train_rmse':float(np.sqrt(np.mean((pf-yf)**2))),
                           'selection_rmse':float(np.sqrt(np.mean((ps-ys)**2)))})
    if not candidates: raise RuntimeError('No usable symbolic equations')
    value={'candidates':candidates,'chosen':choose_equation(candidates),'elapsed_this_session_s':time.time()-t0,'signature':signature}
    save_json(result,value)
    return value

def sr_path(protocol,experiment): return ROOT/'sr'/protocol/experiment

def train_sr_group(protocol,experiment,g,fit,selection,settings,folder):
    filtered=settings['filtered']; cols=feature_cols(fit,usage=settings['usage'])
    if g is not None: cols=[c for c in cols if c!='CHAMBER_GROUP']
    # Remove constants using fitting data only.
    cols=[c for c in cols if fit[c].nunique(dropna=True)>1]
    ranking=rank_features(fit,protocol,cols,folder)
    base=None
    if settings['residual']:
        routed=load_state(base_path(protocol,'PrestonProxy','routed')/'final')
        base=routed['groups'][str(g)] if g is not None else routed
    tuning=[]
    for hp in sr_grid():
        columns=ranking[:min(hp['k'],len(ranking))]; runs=[]
        for seed in CFG['tuning_seeds']:
            runs.append(sr_search(fit,selection,columns,hp,seed,settings['grammar'],
                                  folder/'tuning'/digest(hp)[:12]/str(seed),base))
        tuning.append({'params':hp,'mean_selection_rmse':float(np.mean([r['chosen']['selection_rmse'] for r in runs]))})
    best=min(tuning,key=lambda a:a['mean_selection_rmse'])['params']
    save_json(folder/'tuning.json',{'configurations':tuning,'selected_params':best})
    all_candidates=[]; seed_choices=[]
    for seed in CFG['final_seeds']:
        run=sr_search(fit,selection,ranking[:min(best['k'],len(ranking))],best,seed,settings['grammar'],folder/'final_seeds'/str(seed),base)
        all_candidates.extend(run['candidates']); seed_choices.append(run['chosen'])
    chosen=choose_equation(all_candidates)
    # Full candidate set retained; no training frontier may discard a simpler validated alternative.
    save_json(folder/'candidates.json',all_candidates)
    save_json(folder/'seed_choices.json',seed_choices)
    return {'kind':'symbolic','equation':chosen,'base':base,'seed_choices':seed_choices,
            'fit_ids':sorted(fit.SAMPLE_ID),'selection_ids':sorted(selection.SAMPLE_ID)}

def predict_sr(bundle,d):
    if bundle['kind']=='routed_symbolic':
        p=np.full(len(d),np.nan)
        for g,b in bundle['groups'].items():
            use=(d.CHAMBER_GROUP==int(g)).to_numpy(); p[use]=predict_sr(b,d.loc[use])
        return p
    p=equation_predict(bundle['equation'],d)
    if bundle['base'] is not None: p+=predict_base(bundle['base'],d)
    return p

def fit_symbolic():
    if (ROOT/'sr'/'complete.json').exists(): return
    forbid_training_after_seal()
    for protocol in CFG['protocols']:
        for experiment in CFG['experiments']:
            settings=EXPERIMENTS[experiment]; folder=sr_path(protocol,experiment)
            if load_state(folder/'final') is not None: continue
            fit=frame(protocol,'fit',settings['variant']); selection=frame(protocol,'select',settings['variant'])
            if settings['filtered']: fit=fit[fit[TARGET]<=CFG['flag_threshold']].copy()
            journal('symbolic_experiment_start',protocol=protocol,experiment=experiment,n_fit=len(fit))
            groups={}
            for g in ([0,1] if settings['routed'] else [None]):
                tr=fit if g is None else fit[fit.CHAMBER_GROUP==g]
                va=selection if g is None else selection[selection.CHAMBER_GROUP==g]
                groups[str(g)]=cached(folder/('group_'+str(g))/'chosen',
                    lambda g=g,tr=tr,va=va:train_sr_group(protocol,experiment,g,tr,va,settings,folder/('group_'+str(g))))
            bundle=groups['None'] if 'None' in groups else {'kind':'routed_symbolic','groups':groups}
            bundle.update(protocol=protocol,experiment=experiment,settings=settings,fit_ids=sorted(fit.SAMPLE_ID),selection_ids=sorted(selection.SAMPLE_ID))
            store_state(folder/'final',bundle)
            journal('symbolic_experiment_complete',protocol=protocol,experiment=experiment)
    save_json(ROOT/'sr'/'complete.json',{'complete':True})

def preflight():
    marker=ROOT/'audit'/'preflight.json'
    from pysr import PySRRegressor
    from pysr.julia_import import jl, SymbolicRegression
    versions={'julia':str(jl.seval('VERSION')),'symbolic_regression_jl':str(jl.Base.pkgversion(SymbolicRegression))}
    if marker.exists():
        previous=read_json(marker)
        if any(previous.get(k)!=v for k,v in versions.items()):
            raise RuntimeError('Julia/backend version changed. Do not mix serialized search checkpoints across versions.')
        return
    for grammar in ['arithmetic','bounded','sqrt']:
        rng=np.random.default_rng(123); x=rng.normal(size=(50,2)); y=2*x[:,0]+x[:,1]**2
        model=PySRRegressor(niterations=2,populations=3,population_size=30,ncycles_per_iteration=30,
              maxsize=12,precision=64,random_state=123,deterministic=True,parallelism='serial',
              warm_start=True,progress=False,verbosity=0,batching=False,annealing=True,
              crossover_probability=0.2,input_stream='devnull',
              output_directory=str(ROOT/'audit'/'pysr_preflight'),run_id=grammar,
              **grammar_options(grammar))
        model.fit(x,y,variable_names=['z0','z1'])
        store_state(ROOT/'audit'/('checkpoint_test_'+grammar),{'model':model,'done':2})
        resumed=load_state(ROOT/'audit'/('checkpoint_test_'+grammar))['model']
        if grammar=='sqrt': resumed.set_params(extra_sympy_mappings={'sqrtabs':sqrtabs_sympy})
        resumed.set_params(niterations=1); resumed.fit(x,y,variable_names=['z0','z1'])
        for idx,row in resumed.equations_.iterrows():
            assert np.allclose(expr_predict(str(row.sympy_format),x),resumed.predict(x,index=int(idx)),rtol=1e-8,atol=1e-7)
        versions[grammar]='fit, checkpoint, resume, and expression export passed'
    # A restart must work in a fresh interpreter, not only while Julia functions remain in memory.
    subprocess.run([sys.executable,'-u',str(Path(__file__).resolve()),'--root',str(ROOT),
                    '--config',str(ROOT/'config.json'),'--phase','preflight_reload'],check=True)
    versions['fresh_process_resume']='passed for all three grammars'
    # Preserve resolved Julia project files when available.
    project=Path(str(jl.seval('Base.active_project()')))
    for p in [project,project.with_name('Manifest.toml')]:
        if p.exists(): atomic_bytes(ROOT/'exports'/('Julia_'+p.name),p.read_bytes())
    save_json(marker,versions); journal('preflight_passed',**versions)

def preflight_reload():
    from pysr import PySRRegressor
    rng=np.random.default_rng(123); x=rng.normal(size=(50,2)); y=2*x[:,0]+x[:,1]**2
    for grammar in ['arithmetic','bounded','sqrt']:
        resumed=load_state(ROOT/'audit'/('checkpoint_test_'+grammar))['model']
        if grammar=='sqrt': resumed.set_params(extra_sympy_mappings={'sqrtabs':sqrtabs_sympy})
        resumed.set_params(niterations=1)
        resumed.fit(x,y,variable_names=['z0','z1'])
        for idx,row in resumed.equations_.iterrows():
            actual=resumed.predict(x,index=int(idx))
            exported=expr_predict(str(row.sympy_format),x)
            if not np.allclose(actual,exported,rtol=1e-8,atol=1e-7):
                raise AssertionError('Fresh-process checkpoint export mismatch: '+grammar)
    journal('fresh_process_checkpoint_test_passed')
'''


### Equation checks, complete rediscovery diagnostics, matched-input baselines, calibration and sealing.

In [ ]:
SOURCE_PARTS['05_diagnostics.py'] = '''def symbolic_groups(bundle):
    return bundle['groups'] if bundle['kind']=='routed_symbolic' else {'None':bundle}

def diagnostic_equation(eq,fit,select):
    prep=eq['prep']; n=len(prep['features']); expression=expression_callable(eq['expression'],n)[1]
    mapping={sp.Symbol(f'z{i}'):(sp.Symbol(c)-sp.Float(prep['mu'][i],17))/sp.Float(prep['sd'][i],17)
             for i,c in enumerate(prep['features'])}
    raw=sp.Float(prep['y_mu'],17)+sp.Float(prep['y_sd'],17)*expression.subs(mapping,simultaneous=True)
    # Keep factored form and 17-digit constants; no conversion of protected operators.
    symbols=sorted(raw.free_symbols,key=str); fn=sp.lambdify(symbols,raw,'numpy')
    comparisons={}
    for label,d in [('fit',fit),('select',select)]:
        filled=d[prep['features']].replace([np.inf,-np.inf],np.nan).fillna(dict(zip(prep['features'],prep['median'])))
        rawp=np.broadcast_to(np.asarray(fn(*[filled[str(s)].to_numpy() for s in symbols]),float),(len(d),))
        expected=equation_predict(eq,d)
        if not np.allclose(rawp,expected,rtol=1e-8,atol=1e-7): raise AssertionError('Raw-unit equation export changed predictions')
        comparisons[label]={'max_export_difference':float(np.max(np.abs(rawp-expected))),
                            'negative_predictions':int((expected<0).sum()),'min_prediction':float(expected.min()),'max_prediction':float(expected.max())}
    sensitivity={}; base=equation_predict(eq,select)
    for i,c in enumerate(prep['features']):
        if sp.Symbol(f'z{i}') not in expression.free_symbols: continue
        d=select.copy()
        if c in ['STAGE_B','CHAMBER_GROUP']:
            d[c]=1-d[c]; description='binary-category toggle (not derivative)'
        else:
            d[c]+=0.1*prep['sd'][i]; description='increase by 0.1 training SD; observational sensitivity'
        change=equation_predict(eq,d)-base
        sensitivity[c]={'description':description,'mean_absolute_change':float(np.mean(np.abs(change))),
                        'mean_signed_change':float(np.mean(change))}
    stress={}
    center=pd.DataFrame([dict(zip(prep['features'],prep['median']))]*101)
    for i,c in enumerate(prep['features']):
        if c in ['STAGE_B','CHAMBER_GROUP']: continue
        span=prep['raw_max'][i]-prep['raw_min'][i]
        d=center.copy(); d[c]=np.linspace(prep['raw_min'][i]-.2*span,prep['raw_max'][i]+.2*span,101)
        p=equation_predict(eq,d)
        stress[c]={'nonfinite':int((~np.isfinite(p)).sum()),'negative':int((p<0).sum()),
                   'prediction_min':float(np.nanmin(p)),'prediction_max':float(np.nanmax(p))}
    return {'expression_standardized':eq['expression'],'expression_organizer_units':str(raw),
            'latex_organizer_units':sp.latex(raw),'prep':prep,'complexity_search':eq['complexity'],
            'operations_exported':int(sp.count_ops(raw)),'fit_selection_checks':comparisons,
            'sensitivity':sensitivity,'marginal_stress_20pct_beyond_training':stress,
            'domain_note':'Marginal stress points may not be jointly physically feasible. All inputs are organizer-scaled; no causal claim.'}

def matched_baselines_and_stability():
    if (ROOT/'audit'/'diagnostics_complete.json').exists(): return
    forbid_training_after_seal(); report={}
    for protocol in CFG['protocols']:
        bundle=load_state(sr_path(protocol,'main')/'final')
        fit=frame(protocol,'fit'); select=frame(protocol,'select')
        report[protocol]={}
        for g,sub in symbolic_groups(bundle).items():
            gf=fit if g=='None' else fit[fit.CHAMBER_GROUP==int(g)]
            gs=select if g=='None' else select[select.CHAMBER_GROUP==int(g)]
            eq=sub['equation']; folder=ROOT/'audit'/'equations'/protocol/g
            r=diagnostic_equation(eq,gf,gs)
            expressions=[s['expression'] for s in sub['seed_choices']]
            prep=eq['prep']; frequency={}
            for i,c in enumerate(prep['features']):
                frequency[c]=float(np.mean([sp.Symbol(f'z{i}') in expression_callable(e,len(prep['features']))[1].free_symbols for e in expressions]))
            r['seed_variable_frequency']=frequency
            r['unique_seed_structures']=len(set(expressions))
            r['stability_note']='Search seeds and wafer bootstraps measure different uncertainties; exact strings are only a descriptive structure count.'
            # Re-discover equations after resampling entire fitting wafers. Same fixed search settings.
            boots=[]; ids=gf.WAFER_ID.unique()
            for b in range(CFG['stability_bootstraps']):
                rng=np.random.default_rng(CFG['seed']+b); picked=rng.choice(ids,size=len(ids),replace=True)
                resampled=pd.concat([gf[gf.WAFER_ID==w] for w in picked],ignore_index=True)
                run=sr_search(resampled,gs,eq['prep']['features'],eq['params'],CFG['seed']+b,'arithmetic',folder/'bootstrap_search'/str(b))
                boots.append(run['chosen'])
            r['bootstrap_rediscovered_equations']=boots
            # A separate deletion diagnostic performs a NEW search, not merely linear rescaling.
            errors=np.abs(gf[TARGET].to_numpy()-equation_predict(eq,gf))
            retained=gf.loc[errors<=np.quantile(errors,.99)].copy()
            influence=sr_search(retained,gs,eq['prep']['features'],eq['params'],CFG['final_seeds'][0],
                                'arithmetic',folder/'delete_top_residual_rediscovery')
            r['delete_top1pct_fit_residual_rediscovery']=influence['chosen']
            r['influence_note']='Post-selection exploratory sensitivity on development data, not a final-test competitor.'
            save_json(folder/'diagnostics.json',r); report[protocol][g]=r
        # Models trained on exactly the variables selected by SR, as a controlled comparison.
        for name in ['Ridge','Poly2Ridge','ExtraTrees','SplineRidge']:
            folder=ROOT/'models'/protocol/'matched'/('Matched_'+name+'_routed')
            if load_state(folder/'final') is not None: continue
            groups={}
            for g,sub in symbolic_groups(bundle).items():
                eq=sub['equation']; expression=expression_callable(eq['expression'],len(eq['prep']['features']))[1]
                cols=[c for i,c in enumerate(eq['prep']['features']) if sp.Symbol(f'z{i}') in expression.free_symbols]
                if not cols: cols=['STAGE_B']  # constant-equation edge case, reported explicitly
                tr=fit[fit.CHAMBER_GROUP==int(g)]; va=select[select.CHAMBER_GROUP==int(g)]
                choices=[]
                for hp in parameter_space(name):
                    path=folder/g/digest(hp)[:12]
                    def one(hp=hp,tr=tr,va=va,cols=cols):
                        m=estimator(name,hp).fit(tr[cols],tr[TARGET])
                        return {'model':m,'params':hp,'score':metrics(va[TARGET],m.predict(va[cols]))['RMSE']}
                    choices.append(cached(path,one))
                best=min(choices,key=lambda x:x['score'])
                groups[g]={'kind':'sklearn','estimator':best['model'],'features':cols,'params':best['params']}
            out={'kind':'routed','groups':groups,'name':'Matched_'+name,'mode':'routed','protocol':protocol,'filtered':False,
                 'fit_ids':sorted(fit.SAMPLE_ID),'selection_ids':sorted(select.SAMPLE_ID)}
            store_state(folder/'final',out)
    save_json(ROOT/'audit'/'equation_diagnostics.json',report)
    save_json(ROOT/'audit'/'diagnostics_complete.json',{'complete':True})

def registry():
    out=[]
    for protocol in CFG['protocols']:
        for pointer in sorted((ROOT/'models'/protocol).glob('*/*/final/state.json')):
            folder=pointer.parent; b=load_state(folder)
            label=b['name']+'_'+b['mode']+('_flagged_fit' if b['filtered'] else '')
            out.append({'protocol':protocol,'model':label,'kind':'baseline','path':str(folder.relative_to(ROOT)),'variant':'active'})
        for exp in CFG['experiments']:
            folder=sr_path(protocol,exp)/'final'
            if load_state(folder) is None: raise RuntimeError(f'Missing symbolic model: {protocol}/{exp}')
            out.append({'protocol':protocol,'model':'SR_'+exp,'kind':'symbolic','path':str(folder.relative_to(ROOT)),
                        'variant':EXPERIMENTS[exp]['variant']})
    return out

def predict_entry(entry,d):
    b=load_state(ROOT/entry['path'])
    return predict_sr(b,d) if entry['kind']=='symbolic' else predict_base(b,d)

def conformal_widths(d,p):
    """Group-conditional wafer-max residual calibration. No tuning on calibration."""
    z=d[['WAFER_ID','CHAMBER_GROUP']].copy(); z['error']=np.abs(d[TARGET].to_numpy()-p)
    out={}
    for g,rows in z.groupby('CHAMBER_GROUP'):
        scores=rows.groupby('WAFER_ID').error.max().to_numpy()
        k=int(np.ceil((len(scores)+1)*(1-CFG['interval_alpha'])))
        width=float(np.sort(scores)[k-1]) if k<=len(scores) and np.isfinite(scores).all() else float('inf')
        out[str(int(g))]={'width':width,'n_calibration_wafers':len(scores),'rank':k}
    return out

def seal_models():
    marker=ROOT/'FROZEN.json'
    if marker.exists():
        verify_seal(); return
    for p in [ROOT/'models'/'complete.json',ROOT/'sr'/'complete.json',ROOT/'audit'/'diagnostics_complete.json']:
        if not p.exists(): raise RuntimeError(f'Cannot seal: missing {p.name}')
    entries=registry(); assignments=read_csv(ROOT/'splits'/'assignments.csv'); intervals={}; model_hashes={}
    for entry in entries:
        b=load_state(ROOT/entry['path']); protocol=entry['protocol']
        roles=assignments[assignments.protocol==protocol]
        allowed_fit=set(roles[roles.role=='fit'].SAMPLE_ID); allowed_select=set(roles[roles.role=='select'].SAMPLE_ID)
        assert set(b['fit_ids']).issubset(allowed_fit)
        assert set(b['selection_ids']).issubset(allowed_select)
        assert not (set(b['fit_ids'])|set(b['selection_ids'])) & set(roles[roles.role.isin(['calibrate','test'])].SAMPLE_ID)
        cal=frame(protocol,'calibrate',entry['variant']); pred=predict_entry(entry,cal)
        if not np.isfinite(pred).all(): raise RuntimeError(f'Nonfinite calibration predictions for {entry["model"]}; do not unseal test')
        intervals[protocol+'/'+entry['model']]=conformal_widths(cal,pred)
        state=read_json(ROOT/entry['path']/'state.json'); model_hashes[entry['path']]=state['sha']
    save_json(ROOT/'exports'/'interval_calibration.json',intervals)
    # All competitors, equations, and uncertainty rules now fixed before any holdout labels are joined.
    seal={'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'config_hash':digest(CFG),
          'code_sha256':filehash(__file__),'splits_sha256':filehash(ROOT/'splits'/'assignments.csv'),
          'models':model_hashes,'registry':entries,'intervals_sha256':filehash(ROOT/'exports'/'interval_calibration.json'),
          'prior_exposure_note':'This public dataset and previous results have already been inspected. Computational isolation within this run does not make historical data newly unseen.'}
    save_json(marker,seal); journal('all_models_frozen',n_models=len(entries))

def verify_seal():
    seal=read_json(ROOT/'FROZEN.json')
    assert seal['config_hash']==digest(CFG) and seal['code_sha256']==filehash(__file__)
    assert seal['splits_sha256']==filehash(ROOT/'splits'/'assignments.csv')
    assert seal['intervals_sha256']==filehash(ROOT/'exports'/'interval_calibration.json')
    for folder,sha in seal['models'].items():
        assert read_json(ROOT/folder/'state.json')['sha']==sha
        assert filehash(ROOT/folder/(sha+'.pkl'))==sha
    return seal
'''


### Locked evaluation, uncertainty, drift, plots and verified portable equation exports.

In [ ]:
SOURCE_PARTS['06_evaluation.py'] = '''def prediction_file(entry):
    return ROOT/'exports'/'predictions'/entry['protocol']/(entry['model']+'.csv')

def evaluate_frozen():
    seal=verify_seal(); intervals=read_json(ROOT/'exports'/'interval_calibration.json')
    marker=ROOT/'exports'/'evaluation_complete.json'
    if marker.exists(): return
    # A resumed evaluation may re-read labels only to finish the SAME frozen plan.
    journal('evaluation_started_or_resumed',frozen_sha256=filehash(ROOT/'FROZEN.json'))
    rows=[]
    for entry in seal['registry']:
        path=prediction_file(entry)
        if path.exists(): d=read_csv(path)
        else:
            truth=frame(entry['protocol'],'test',entry['variant'])
            p=predict_entry(entry,truth)
            d=truth[['SAMPLE_ID','WAFER_ID','STAGE','CHAMBER_GROUP','T_START',TARGET]].copy(); d['prediction']=p
            widths=intervals[entry['protocol']+'/'+entry['model']]
            d['interval_halfwidth']=d.CHAMBER_GROUP.map(lambda g:widths[str(int(g))]['width'])
            d['lower']=d.prediction-d.interval_halfwidth; d['upper']=d.prediction+d.interval_halfwidth
            # No clipping or replacement of invalid predictions based on test labels.
            d['covered']=(d[TARGET]>=d.lower)&(d[TARGET]<=d.upper)
            save_csv(path,d)
        subsets={'all':d}
        subsets.update({'chamber_'+str(int(g)):q for g,q in d.groupby('CHAMBER_GROUP')})
        subsets.update({'stage_'+str(s):q for s,q in d.groupby('STAGE')})
        subsets['label_at_most_flag_threshold']=d[d[TARGET]<=CFG['flag_threshold']]
        if (d[TARGET]>CFG['flag_threshold']).any(): subsets['flagged_labels']=d[d[TARGET]>CFG['flag_threshold']]
        for subset,q in subsets.items():
            if not len(q): continue
            m=metrics(q[TARGET],q.prediction)
            rows.append({'protocol':entry['protocol'],'model':entry['model'],'subset':subset,**m,
                'row_interval_coverage':float(q.covered.mean()),
                'wafer_simultaneous_coverage':float(q.groupby('WAFER_ID').covered.all().mean()),
                'mean_interval_width':float((q.upper-q.lower).mean())})
    save_csv(ROOT/'tables'/'test_metrics.csv',pd.DataFrame(rows))
    # Compare mirror answer keys only NOW, without implying their independent provenance is verified.
    base=extract_data(); comparisons={}
    for part,filename in [('test','orig_CMP-test-removalrate.csv'),('val','orig_CMP-validation-removalrate.csv')]:
        candidates=list(base.rglob(filename))
        if candidates:
            answers=pd.read_csv(candidates[0],dtype={'WAFER_ID':str,'STAGE':str}).sort_values(KEY).reset_index(drop=True)
            labels=load_labels(part)[KEY+[TARGET]].sort_values(KEY).reset_index(drop=True)
            comparisons[part]={'mirror_matches_answer_key_file':bool(labels.equals(answers[KEY+[TARGET]])),
                               'key_path':str(candidates[0].relative_to(base))}
    save_json(ROOT/'audit'/'answer_key_comparison.json',comparisons)
    save_json(marker,{'frozen_sha256':filehash(ROOT/'FROZEN.json'),'n_model_evaluations':len(seal['registry']),
                      'predictions':{str(prediction_file(e).relative_to(ROOT)):filehash(prediction_file(e)) for e in seal['registry']}})

def bootstrap_differences(d,comparators,temporal=False):
    # Aggregate per wafer; blocks preserve local dependence in chronological evaluation.
    groups=d.groupby('WAFER_ID',sort=False)
    keys=groups.T_START.min().sort_values().index.tolist()
    counts=groups.size().reindex(keys).to_numpy(float)
    sums={c:groups.apply(lambda q,c=c:float(np.sum((q[c]-q[TARGET])**2)),include_groups=False).reindex(keys).to_numpy() for c in ['SR_main']+comparators}
    rng=np.random.default_rng(CFG['seed']); n=len(keys); draws={c:[] for c in comparators}; sr_draws=[]
    block=max(2,int(np.ceil(np.sqrt(n))))
    for _ in range(CFG['bootstrap_draws']):
        if temporal:
            starts=rng.integers(0,n,size=int(np.ceil(n/block)))
            idx=np.concatenate([(s+np.arange(block))%n for s in starts])[:n]
        else: idx=rng.integers(0,n,size=n)
        total=counts[idx].sum(); sr=float(np.sqrt(sums['SR_main'][idx].sum()/total)); sr_draws.append(sr)
        for c in comparators: draws[c].append(sr-float(np.sqrt(sums[c][idx].sum()/total)))
    rows=[]; point=metrics(d[TARGET],d.SR_main)['RMSE']; m=max(1,len(comparators))
    for c,a in draws.items():
        lo,hi=np.quantile(a,[.025,.975]); blo,bhi=np.quantile(a,[.025/m,1-.025/m])
        rows.append({'comparison':'SR_main minus '+c,'difference_rmse':point-metrics(d[TARGET],d[c])['RMSE'],
                     'ci95_low':float(lo),'ci95_high':float(hi),'family_adjusted_low':float(blo),'family_adjusted_high':float(bhi),
                     'fraction_bootstrap_differences_positive':float(np.mean(np.asarray(a)>0)),
                     'method':'circular moving blocks of chronological wafer groups' if temporal else 'paired wafer-cluster percentile bootstrap',
                     'block_length':block if temporal else 1,'n_wafer_groups':n})
    return rows,{'rmse':point,'ci95':np.quantile(sr_draws,[.025,.975]).tolist()}

def uncertainty_and_shift():
    marker=ROOT/'exports'/'uncertainty_complete.json'
    if marker.exists(): return
    seal=verify_seal(); comparisons=[]; maincis={}; shifts=[]
    for protocol in CFG['protocols']:
        entries=[e for e in seal['registry'] if e['protocol']==protocol]
        base=read_csv(prediction_file(next(e for e in entries if e['model']=='SR_main')))
        base=base.rename(columns={'prediction':'SR_main'})
        candidates=['ExtraTrees_routed','HistGradientBoosting_routed','Ridge_routed','ElasticNet_routed',
                    'PrestonProxy_routed','GroupMean_routed','Matched_Poly2Ridge_routed','Matched_SplineRidge_routed']
        present=[]
        for name in candidates:
            matching=[e for e in entries if e['model']==name]
            if not matching: continue
            q=read_csv(prediction_file(matching[0]))[['SAMPLE_ID','prediction']].rename(columns={'prediction':name})
            base=checked_merge(base,q,['SAMPLE_ID']); present.append(name)
        if not np.isfinite(base[['SR_main']+present].to_numpy()).all():
            maincis[protocol]={'status':'invalid predictions; comparisons not computed'}; continue
        rs,ci=bootstrap_differences(base,present,protocol=='chronological')
        for r in rs: r['protocol']=protocol
        comparisons+=rs; maincis[protocol]=ci
        # Report ordinary wafer bootstrap too for temporal sensitivity, clearly distinguished.
        if protocol=='chronological':
            rs,_=bootstrap_differences(base,present,False)
            for r in rs: r['protocol']=protocol; r['method']+=' (sensitivity, ignores serial dependence)'
            comparisons+=rs
        for subset,q in base.groupby('CHAMBER_GROUP'):
            _,ci=bootstrap_differences(q,[],protocol=='chronological'); maincis[protocol+'/chamber_'+str(int(subset))]=ci
        fit=frame(protocol,'fit'); test=frame(protocol,'test')
        b=load_state(sr_path(protocol,'main')/'final')
        for g,sub in symbolic_groups(b).items():
            tr=fit[fit.CHAMBER_GROUP==int(g)]; te=test[test.CHAMBER_GROUP==int(g)]
            prep=sub['equation']['prep']; z=transform(te,prep)
            outside_any=np.zeros(len(te),bool)
            for i,c in enumerate(prep['features']):
                outside=(te[c]<prep['raw_min'][i])|(te[c]>prep['raw_max'][i]); outside_any|=outside.to_numpy()
                shifts.append({'protocol':protocol,'group':g,'feature':c,'train_min':prep['raw_min'][i],
                    'train_max':prep['raw_max'][i],'test_min':float(te[c].min()),'test_max':float(te[c].max()),
                    'fraction_outside_training_range':float(outside.mean())})
            maincis[protocol+'/chamber_'+g]['fraction_any_input_outside_training_range']=float(outside_any.mean())
    save_csv(ROOT/'tables'/'paired_comparisons.csv',pd.DataFrame(comparisons))
    save_csv(ROOT/'tables'/'input_shift.csv',pd.DataFrame(shifts))
    save_json(ROOT/'tables'/'rmse_intervals.json',maincis)
    save_json(marker,{'complete':True,'bootstrap_draws':CFG['bootstrap_draws']})

def publication_outputs():
    verify_seal(); table=read_csv(ROOT/'tables'/'test_metrics.csv')
    eqreport=read_json(ROOT/'audit'/'equation_diagnostics.json'); equations=[]
    for protocol,groups in eqreport.items():
        for g,r in groups.items():
            equations.append({'protocol':protocol,'chamber_group':g,'expression':r['expression_organizer_units'],
                              'search_complexity':r['complexity_search'],'exported_operations':r['operations_exported']})
    save_csv(ROOT/'exports'/'equations.csv',pd.DataFrame(equations))
    # Portable JSON plus a pure NumPy/SymPy evaluator; no Julia is needed for deployed main equations.
    portable={}
    for protocol in CFG['protocols']:
        b=load_state(sr_path(protocol,'main')/'final')
        portable[protocol]={g:sub['equation'] for g,sub in symbolic_groups(b).items()}
    save_json(ROOT/'exports'/'main_equations.json',portable)
    evaluator=\'\'\'import json\\nimport numpy as np\\nimport sympy as sp\\n\\ndef predict(features, protocol, equations_path="main_equations.json"):\\n    equations=json.load(open(equations_path))\\n    output=np.full(len(features),np.nan)\\n    for group,eq in equations[protocol].items():\\n        mask=(features["CHAMBER_GROUP"].to_numpy()==int(group))\\n        prep=eq["prep"]; x=features.loc[mask,prep["features"]].to_numpy(float)\\n        x=np.where(np.isfinite(x),x,np.asarray(prep["median"]))\\n        z=(x-np.asarray(prep["mu"]))/np.asarray(prep["sd"])\\n        names=sp.symbols(" ".join("z"+str(i) for i in range(z.shape[1])),seq=True)\\n        fn=sp.lambdify(names,sp.sympify(eq["expression"]),"numpy")\\n        output[mask]=prep["y_mu"]+prep["y_sd"]*np.broadcast_to(fn(*z.T),(len(z),))\\n    if not np.isfinite(output).all():\\n        raise ValueError("Unknown chamber group or nonfinite equation output")\\n    return output\\n\'\'\'
    atomic_bytes(ROOT/'exports'/'predict_equations.py',evaluator.encode())
    namespace={}; exec(compile(evaluator,'portable_evaluator','exec'),namespace)
    verification=[]
    for protocol in CFG['protocols']:
        d=frame(protocol,'test'); portable_pred=namespace['predict'](d,protocol,str(ROOT/'exports'/'main_equations.json'))
        saved=read_csv(ROOT/'exports'/'predictions'/protocol/'SR_main.csv')
        aligned=checked_merge(d[['SAMPLE_ID']],saved[['SAMPLE_ID','prediction']],['SAMPLE_ID'])
        err=float(np.max(np.abs(portable_pred-aligned.prediction.to_numpy())))
        if err>1e-7: raise AssertionError('Portable evaluator does not reproduce test predictions')
        verification.append({'protocol':protocol,'max_difference':err})
    save_json(ROOT/'exports'/'portable_evaluator_verification.json',verification)
    # Figures are generated from saved predictions, never used to change the frozen models.
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
    def savefig(name):
        plt.tight_layout(); plt.savefig(ROOT/'figures'/(name+'.png'),dpi=220,bbox_inches='tight')
        plt.savefig(ROOT/'figures'/(name+'.pdf'),bbox_inches='tight'); plt.close()
    traj=read_csv(ROOT/'data'/'representative_trajectories.csv')
    fig,axes=plt.subplots(2,2,figsize=(10,6))
    for j,g in enumerate([0,1]):
        d=traj[traj.CHAMBER_GROUP==g].sort_values('TIMESTAMP'); t=d.TIMESTAMP-d.TIMESTAMP.min()
        for i,c in enumerate(['CENTER_AIR_BAG_PRESSURE','STAGE_ROTATION']):
            axes[i,j].plot(t,d[c],lw=.8); axes[i,j].set_title(f'Group {g}: {c}'); axes[i,j].set_xlabel('Recorded relative time')
    savefig('01_development_trajectories')
    for protocol in CFG['protocols']:
        fit=frame(protocol,'fit'); fig,ax=plt.subplots(figsize=(8,4))
        for g,d in fit.groupby('CHAMBER_GROUP'): ax.hist(d[TARGET],bins=40,alpha=.6,label=f'Group {g}')
        ax.set_yscale('log'); ax.set_xlabel('MRR (all fitting labels retained)'); ax.set_ylabel('Count, log scale'); ax.legend()
        savefig(protocol+'_02_label_distribution')
        main=read_csv(ROOT/'exports'/'predictions'/protocol/'SR_main.csv')
        fig,axes=plt.subplots(1,2,figsize=(10,4))
        for ax,(g,d) in zip(axes,main.groupby('CHAMBER_GROUP')):
            ax.scatter(d[TARGET],d.prediction,s=12,alpha=.65); lo=min(d[TARGET].min(),d.prediction.min()); hi=max(d[TARGET].max(),d.prediction.max())
            ax.plot([lo,hi],[lo,hi],'k--',lw=1); ax.set_title(f'Group {g}, n={len(d)}'); ax.set_xlabel('Measured MRR'); ax.set_ylabel('Predicted MRR')
        savefig(protocol+'_03_predictions')
        d=main.sort_values('T_START'); plt.figure(figsize=(9,4)); plt.scatter(d.T_START,d[TARGET]-d.prediction,c=d.CHAMBER_GROUP,s=12,cmap='coolwarm'); plt.axhline(0,c='k',lw=1)
        plt.xlabel('Recorded time'); plt.ylabel('MRR residual'); savefig(protocol+'_04_residual_time')
        sub=table[(table.protocol==protocol)&(table.subset=='all')].sort_values('RMSE')
        plt.figure(figsize=(10,max(5,len(sub)*.23))); plt.barh(sub.model,sub.RMSE); plt.xlabel('Holdout RMSE'); savefig(protocol+'_05_models_and_ablations')
        fig,axes=plt.subplots(1,2,figsize=(10,4))
        for ax,g in zip(axes,['0','1']):
            cands=read_json(sr_path(protocol,'main')/('group_'+g)/'candidates.json')
            q=pd.DataFrame(cands); ax.scatter(q.complexity,q.selection_rmse,s=8,alpha=.3)
            best=q.groupby('complexity').selection_rmse.min().sort_index(); ax.plot(best.index,best.values,lw=1)
            ax.set_title('Group '+g); ax.set_xlabel('Search-tree complexity'); ax.set_ylabel('Selection RMSE')
        savefig(protocol+'_06_complexity')
    # Plain results report: no invented interpretation, acceptance claims, or literature section.
    lines=['# CMP symbolic regression results','',
           'Generated from frozen model artifacts. All primary fitting labels retained. Flagged-fit sensitivity removes only fitting labels above the configured threshold.',
           '','## Overall holdout results','',table[table.subset=='all'].to_csv(index=False),
           '## Interpretation limits','',
           '- Official partitions may share wafers across stages. Only the wafer-disjoint protocol assesses entirely held-out wafer IDs.',
           '- Chronological preprocessing, rankings and tuning use earlier roles only; crossing wafers are purged.',
           '- All models fit on the same fit role; selection data are not included in a post-selection refit.',
           '- Bootstrap intervals describe fixed-model evaluation uncertainty, not all model-development uncertainty.',
           '- Time-block intervals are a dependence sensitivity analysis; results from one process remain limited.',
           '- Conformal coverage relies on exchangeability; temporal drift and the overlapping official protocol violate or weaken that premise. Empirical coverage is reported.',
           '- Channels have undisclosed scaling. Usage counters are predictive associations, not identified causal wear effects.',
           '- High-activity segmentation is a transparent heuristic, not a validated process-state label.',
           '- This public dataset has already been explored. Do not present repeated use as a newly untouched external test.',
           '- Search and ablation budgets are recorded, equal within symbolic experiments; matched-input baselines are included.',
           '- No acceptance or novelty claim is produced automatically.']
    atomic_bytes(ROOT/'exports'/'RESULTS.md','\\n'.join(lines).encode())
    save_json(ROOT/'exports'/'COMPLETED.json',{'all_stages_complete':True,'frozen_sha256':filehash(ROOT/'FROZEN.json')})
    # A single downloadable archive of outputs, without raw/reconstructed source data or large checkpoints.
    zip_path=ROOT/'CMP_results.zip'
    with zipfile.ZipFile(zip_path.with_suffix('.pending'),'w',zipfile.ZIP_DEFLATED) as z:
        for folder in ['tables','figures','exports','audit','splits']:
            for p in (ROOT/folder).rglob('*'):
                if p.is_file() and p.suffix in ['.csv','.json','.md','.png','.pdf','.py','.toml'] and 'bootstrap_search' not in str(p) and 'pysr_preflight' not in str(p):
                    z.write(p,p.relative_to(ROOT))
        for name in ['configuration.json','environment.json','FROZEN.json','pipeline.py','config.json','requirements.lock.txt']:
            if (ROOT/name).exists(): z.write(ROOT/name,name)
    os.replace(zip_path.with_suffix('.pending'),zip_path)
    journal('all_outputs_complete',archive=str(zip_path))

def main():
    parser=argparse.ArgumentParser(); parser.add_argument('--root',required=True); parser.add_argument('--config',required=True)
    parser.add_argument('--phase',required=True,choices=['preflight','preflight_reload','data','splits','baselines','symbolic','diagnostics','freeze','evaluate','uncertainty','outputs'])
    args=parser.parse_args(); init(args.root,args.config)
    funcs={'preflight':preflight,'preflight_reload':preflight_reload,'data':build_features,'splits':make_splits,'baselines':fit_baselines,
           'symbolic':fit_symbolic,'diagnostics':matched_baselines_and_stability,'freeze':seal_models,
           'evaluate':evaluate_frozen,'uncertainty':uncertainty_and_shift,'outputs':publication_outputs}
    try:
        journal('phase_started',phase=args.phase); funcs[args.phase](); journal('phase_completed',phase=args.phase)
    except Exception as e:
        journal('phase_failed',phase=args.phase,error=repr(e)); traceback.print_exc(); raise

if __name__=='__main__': main()
'''


## 4. Assemble, validate syntax, and run the engine preflight

The preflight performs small real PySR searches for all operator sets, saves and reloads checkpoints, resumes searches in both the original and a **fresh Python process**, and checks that exported expressions reproduce PySR predictions. It fails before the expensive study if the environment or engine is incompatible.


In [11]:
import ast
PIPELINE = '\n\n'.join(SOURCE_PARTS[k] for k in sorted(SOURCE_PARTS))
ast.parse(PIPELINE)
pipeline_path = STUDY / 'pipeline.py'
if pipeline_path.exists() and pipeline_path.read_text(encoding='utf-8') != PIPELINE:
    raise RuntimeError('Implementation changed for this study. Use the original notebook to resume, or a new STUDY_NAME.')
pipeline_path.write_text(PIPELINE, encoding='utf-8')
(STUDY / 'logs').mkdir(exist_ok=True)

def run_phase(phase):
    command = [str(PYTHON), '-u', str(pipeline_path), '--root', str(STUDY),
               '--config', str(config_path), '--phase', phase]
    log = STUDY / 'logs' / (phase + '.log')
    print('Running:', phase, ' | persistent log:', log, flush=True)
    with open(log, 'a', encoding='utf-8') as logfile:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env=WORKER_ENV)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
                logfile.write(line); logfile.flush()
            status = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            try: proc.wait(timeout=15)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            raise
    if status:
        raise RuntimeError(f'Phase {phase} failed (exit {status}). Its completed checkpoints are preserved. See {log}.')

run_phase('preflight')
if IN_COLAB and (not JULIA_CACHE.exists() or not JULIA_CACHE_META.exists()):
    print('Saving Julia package cache to Drive for future resumes...')
    pending_cache = JULIA_CACHE.with_suffix('.partial')
    try:
        with tarfile.open(pending_cache, 'w:gz', compresslevel=3) as cached_julia:
            for name in ['julia_depot', 'julia_project']:
                if (LOCAL / name).exists():
                    cached_julia.add(LOCAL / name, arcname=name,
                        filter=lambda info: None if info.name.endswith(('.pid', '.pidfile', '.lock')) else info)
        os.replace(pending_cache, JULIA_CACHE)
        atomic_text(JULIA_CACHE_META, json.dumps({'sha256': sha256_file(JULIA_CACHE)}))
    except (OSError, tarfile.TarError) as exc:
        print('Optional Julia package-cache backup failed:', repr(exc), flush=True)
        print('The study will continue. Model checkpoints are saved separately; backend packages may need '
              'downloading again after reconnecting. Check Drive space if later checkpoint writes fail.', flush=True)


Running: preflight  | persistent log: /content/drive/MyDrive/CMP_SR_Research/CMP_SR_Colab_native_full/logs/preflight.log
2026-09-28T13:49:24Z phase_started {"phase":"preflight"}
/usr/local/lib/python3.13/dist-packages/pysr/julia_import.py:39: UserWarning: PYTHON_JULIACALL_THREADS environment variable is set to something other than 'auto', so PySR was not able to set it. You may wish to set it to `'auto'` for full use of your CPU.
  warnings.warn(
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: juliapkg_exe=/usr/local/bin/julia is Julia 1.12.6 but 1.10.3 - 1.11 is required.
[juliapkg] Using Julia 1.12.6 at /usr/local/bin/julia
[juliapkg] Finding libjulia and bindir:
           | using Libdl
          

RuntimeError: Phase preflight failed (exit 1). Its completed checkpoints are preserved. See /content/drive/MyDrive/CMP_SR_Research/CMP_SR_Colab_native_full/logs/preflight.log.

## 5. Download and audit the dataset; build both feature representations

Only original sensor data and label files are used. Exact duplicate sensor rows are removed and counted. The released data also contain simultaneous measurement updates: within each chamber and ring location, numeric readings at the same timestamp are averaged and the collapsed count is recorded. Joins must be one-to-one and complete. Chamber routing uses sensor context, never a target threshold.

The main activity interval is a **run-local high-activity heuristic** combining available pressure and rotation channels separately within each chamber. It is not claimed to be ground-truth polishing segmentation. Whole-run features form a full ablation. Long gaps are excluded from integrations and recorded. Fitting-only median imputation handles missing numerical features.

All primary labels remain in the analysis. Labels above 1000 are flagged, not declared invalid. A separate experiment excludes such labels **only from fitting**, while every evaluation label is retained.


In [ ]:
run_phase('data')


## 6. Freeze role assignments and verify separation

| Protocol | Fitting | Selection/calibration | Final evaluation |
|---|---|---|---|
| Official benchmark | Official training | Official validation split by wafer: 75% selection, 25% calibration | Official test; cross-partition wafer overlap explicitly reported |
| Wafer-disjoint | Training wafers excluding all validation/test wafers | Validation wafers excluding test wafers; then wafer-grouped selection/calibration | Official test, entirely disjoint wafer IDs from fitting/selection/calibration |
| Chronological | Earliest 60% boundary | Next 15% selection, next 10% calibration | Latest 15% boundary, within official train+validation pool |

Chronological boundaries are based on wafer timing; wafers whose full trajectories cross boundaries are purged. Assertions check both wafer disjointness and strictly increasing end/start times. Internal chronological CV is also forward-only. No ranking or tuned settings transfer from another protocol.

Selection chooses models. Calibration is used only for intervals. All competing methods fit on the same fitting role; none refits on the selection role before evaluation.


In [ ]:
run_phase('splits')
print((STUDY / 'splits' / 'report.json').read_text())


## 7. Tune and checkpoint every baseline

Hyperparameters are searched using grouped CV or forward CV entirely within the fitting role. Up to three configurations per model are shortlisted using CV, then selected on the selection role. CV scores are tuning estimates; they are not reported as final generalization performance. Global and chamber-routed variants are included, along with a separately labeled flagged-fitting-label sensitivity.


In [ ]:
run_phase('baselines')


## 8. Tune PySR and run all symbolic experiments

Each experiment uses the same configuration count, tuning seeds, final seeds, and iterations. Each has its own feature ranking and tuning. The simplest candidate within 5% of the best selection RMSE is chosen from all finite candidates; the selection set, not the test set, determines this tradeoff.

Arithmetic uses `+`, `−`, `×`; bounded operators add `tanh`; the square-root experiment uses `sqrt(abs(x))` with an exactly matching SymPy representation. The physics-correction experiment adds SR predictions to a fitted pressure–rotation proxy. The proxy does not assume unknown physical units.

Standardization and imputation are learned on the fitting role. Checkpoints are saved every 10 completed search iterations. If Colab dies during a chunk, only that unfinished chunk is repeated.


In [ ]:
run_phase('symbolic')


## 9. Equation checks, complete rediscovery, and matched-variable comparisons

This stage verifies full-precision raw-input expressions, numerical domains, prediction ranges on development data, category toggles, continuous sensitivity, and marginal extrapolation stress. It records seed variability and reruns equation discovery after wafer-level resampling and deletion of the largest 1% of fitting residuals.

These are descriptive robustness checks, not evidence of causality. Additional Ridge, polynomial, spline, and Extra Trees models use exactly the variables selected by the main symbolic equation.


In [ ]:
run_phase('diagnostics')


## 10. Calibrate intervals and seal every model before opening holdouts

Calibration uses each wafer's maximum absolute residual within a chamber group, producing group-conditional intervals. Exchangeability assumptions are questionable under temporal drift and official wafer overlap; empirical coverage will therefore be reported explicitly.

The seal contains every model hash, code/config fingerprint, split hash, and the full evaluation registry. Fitting and selection sample IDs are checked against their permitted roles. All primary models and ablations are sealed together before any final-test scoring. Subsequent training is blocked for this study.


In [ ]:
run_phase('freeze')
print('All model choices are now frozen.')


## 11. Execute the frozen holdout evaluations

Metrics include RMSE, MSE, MAE and R² overall, by chamber group, by stage, and for clearly labeled target-range diagnostic subsets. Overall R² can be dominated by between-group differences; always inspect subgroup metrics. No test observations are removed from the primary scores.

Resuming a partial evaluation continues the same sealed plan. It never triggers retuning or silently substitutes a different model.


In [ ]:
run_phase('evaluate')


## 12. Paired uncertainty and distribution shift

The notebook reports paired wafer-cluster bootstrap intervals for SR-minus-baseline RMSE differences. Chronological results also use circular moving blocks of chronologically ordered wafers. Pointwise and multiplicity-adjusted intervals are labeled separately. Bootstrap positive fractions are descriptive resampling fractions, not p-values.

Input-range drift is measured from the fitting distribution to each holdout, including the fraction outside the training domain. Conformal calibration rows do not influence rankings or model selection.


In [ ]:
run_phase('uncertainty')


## 13. Generate outputs and verify standalone predictions

All figures and tables are built from saved predictions. The notebook exports full-precision equations, preprocessing constants, a Julia-free NumPy/SymPy predictor, seed and bootstrap diagnostics, immutable split assignments, environment locks, and per-sample predictions. The exported predictor must reproduce the saved main-model predictions within tolerance.

Raw source data and large training checkpoints are excluded from the results archive. Check original dataset terms before redistributing derived data or predictions publicly.


In [ ]:
run_phase('outputs')


## 14. View results and download

Your complete persistent study stays in Google Drive. The archive below contains the results; all checkpoints remain in the study folder for resuming and auditing.


In [ ]:
metrics_path = STUDY / 'tables' / 'test_metrics.csv'
with open(metrics_path, newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['subset'] == 'all']
columns = ['protocol', 'model', 'n', 'RMSE', 'MAE', 'R2', 'row_interval_coverage']
head = '<tr>' + ''.join('<th>'+html.escape(c)+'</th>' for c in columns) + '</tr>'
body = ''.join('<tr>'+''.join('<td>'+html.escape(r.get(c,''))+'</td>' for c in columns)+'</tr>' for r in rows)
display(HTML('<h3>Frozen holdout results</h3><table>'+head+body+'</table>'))
for filename in ['wafer_disjoint_03_predictions.png', 'chronological_04_residual_time.png', 'wafer_disjoint_06_complexity.png']:
    p = STUDY / 'figures' / filename
    if p.exists(): display(Image(filename=str(p), width=900))
print('Completed:', STUDY)
print('All results:', STUDY / 'CMP_results.zip')
print('Final status:', (STUDY / 'exports' / 'COMPLETED.json').read_text())
if IN_COLAB:
    from google.colab import files
    files.download(str(STUDY / 'CMP_results.zip'))
else:
    display(FileLink(str(STUDY / 'CMP_results.zip')))


## Resume checklist

1. Reopen this same notebook and reconnect to a compatible CPU runtime.
2. Keep `STUDY_NAME` and configuration unchanged, then click **Run all**.
3. Authorize the same Google Drive account. Finished stages load from disk; interrupted searches resume at their last complete chunk.

Do not delete `FROZEN.json`, edit checkpoints, or run two copies against the same study directory. A new method requires a new study directory and an explicit acknowledgment that previously examined holdouts have been reused.

### Compatibility and verification

This complete notebook uses the Python and scientific-library versions present in your Colab runtime. Only PySR and missing dependencies are installed. Before/after version records verify that existing dependencies were preserved. All analysis stages and full default budgets remain included.

The analysis implementation has passed real-data extraction, feature and split checks, reduced-budget baseline runs, expression checks, and a workflow test using a mock symbolic engine. That mock was used only for local testing; it is not in this notebook. The native setup's interpreter selection, dependency constraints, successful-resume path, changed-version guard, installation-failure handling, and unexpected-package-change guard passed offline tests with simulated installer results. These are startup-logic tests, not a real package installation or a live Colab run. Every code cell is syntax-checked and the notebook structure is validated before delivery.

Recheck on 28 September 2026: PySR's dependencies resolved for Linux/Python 3.13 while retaining the versions in Google's published Colab package inventory. Added-dependency restoration and changed-backend guards passed offline startup tests. The optional Julia cache backup can now fail without blocking analysis; model checkpoint failures remain fatal and visible.

The complete notebook has **not been executed in a live Colab runtime before delivery**. A new attempt to run real PySR locally was blocked during Julia package installation by an upstream registry HTTP 403; the Windows host had also reported TLS credential errors. No real PySR fit is claimed from those attempts. Real searches still require the mandatory Colab preflight. That preflight fits models, verifies prediction exports, and reloads/resumes checkpoints in a fresh process before long training begins. A passed preflight establishes that those operations work in your current runtime; it does not guarantee external services, unlimited runtime duration, or scientific quality of the final equations.

Official software references: [PySR 2.5.0](https://pypi.org/project/pysr/2.5.0/), [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html), [Colab's published package inventory](https://github.com/googlecolab/backend-info/blob/master/pip-freeze.txt). The runtime's actual versions are checked at execution; the published inventory is only a reference.

